# TODO: Refactor this notebook 

# Image processing pipeline to extract some feature (i tried some stuff)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure
import os

def process_image_pipeline(image_path):
    """
    Complete image processing pipeline: HOG, saliency map, and segmentation
    """
    # Read the TIFF file
    if not os.path.exists(image_path):
        print(f"Error: File {image_path} not found!")
        return None
    
    img = cv2.imread(image_path)
    if img is None:
        # Try alternative reading method for some TIFF formats
        img = cv2.imread(image_path, cv2.IMREAD_ANYCOLOR)
    if img is None:
        print(f"Error: Could not read {image_path} as image!")
        return None
        
    print(f"Successfully loaded image with shape: {img.shape}")
    
    # Convert to RGB for proper color display
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Create visualization figure
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Original Image
    plt.subplot(3, 4, 1)
    plt.imshow(img_rgb)
    plt.title('1. Original Color Image')
    plt.axis('off')
    
    # 2. Grayscale Image
    plt.subplot(3, 4, 2)
    plt.imshow(img_gray, cmap='gray')
    plt.title('2. Grayscale Image')
    plt.axis('off')
    
    # 3. Compute HOG features and visualization :cite[1]:cite[5]
    orientations = 9
    pixels_per_cell = (8, 8)
    cells_per_block = (2, 2)
    
    try:
        hog_features, hog_image = hog(img_gray, 
                                    orientations=orientations,
                                    pixels_per_cell=pixels_per_cell,
                                    cells_per_block=cells_per_block,
                                    visualize=True,
                                    block_norm='L2-Hys')
        
        # Enhance HOG visualization for better contrast :cite[1]
        hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
        
        plt.subplot(3, 4, 3)
        plt.imshow(hog_image_rescaled, cmap='viridis')
        plt.title('3. HOG Features Visualization')
        plt.axis('off')
        
        plt.subplot(3, 4, 4)
        plt.imshow(hog_image_rescaled, cmap='inferno')
        plt.title('4. HOG (Alternative Colormap)')
        plt.axis('off')
        
        print(f"HOG features extracted: {len(hog_features)} dimensions")
        
    except Exception as e:
        print(f"Error in HOG computation: {e}")
        return None
    
    # 4. Compute Saliency Map using spectral residual approach :cite[2]:cite[6]
    saliency_map = compute_saliency_map(img_gray)
    
    plt.subplot(3, 4, 5)
    plt.imshow(saliency_map, cmap='hot')
    plt.title('5. Saliency Map (Heat)')
    plt.axis('off')
    
    plt.subplot(3, 4, 6)
    plt.imshow(saliency_map, cmap='gray')
    plt.title('6. Saliency Map (Grayscale)')
    plt.axis('off')
    
    # 5. Overlay saliency on original image
    plt.subplot(3, 4, 7)
    plt.imshow(img_rgb)
    plt.imshow(saliency_map, cmap='jet', alpha=0.5)
    plt.title('7. Saliency Overlay on Original')
    plt.axis('off')
    
    # 6. Image Binarization and Segmentation :cite[3]:cite[10]
    # Method 1: Otsu's thresholding
    _, binary_otsu = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    plt.subplot(3, 4, 8)
    plt.imshow(binary_otsu, cmap='gray')
    plt.title('8. Otsu Binary Segmentation')
    plt.axis('off')
    
    # Method 2: Adaptive thresholding :cite[3]
    binary_adaptive = cv2.adaptiveThreshold(img_gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv2.THRESH_BINARY, 11, 2)
    
    plt.subplot(3, 4, 9)
    plt.imshow(binary_adaptive, cmap='gray')
    plt.title('9. Adaptive Binary Segmentation')
    plt.axis('off')
    
    # Method 3: Saliency-based segmentation
    _, binary_saliency = cv2.threshold((saliency_map * 255).astype(np.uint8), 
                                     0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    plt.subplot(3, 4, 10)
    plt.imshow(binary_saliency, cmap='gray')
    plt.title('10. Saliency-based Segmentation')
    plt.axis('off')
    
    # 7. Combined segmentation (Otsu + Saliency)
    combined_segmentation = cv2.bitwise_and(binary_otsu, binary_saliency)
    
    plt.subplot(3, 4, 11)
    plt.imshow(combined_segmentation, cmap='gray')
    plt.title('11. Combined Segmentation')
    plt.axis('off')
    
    # 8. Overlay segmentation on original
    plt.subplot(3, 4, 12)
    plt.imshow(img_rgb)
    # Create colored segmentation overlay
    segmentation_overlay = np.zeros_like(img_rgb)
    segmentation_overlay[combined_segmentation > 0] = [255, 0, 0]  # Red color
    plt.imshow(segmentation_overlay, alpha=0.3)
    plt.title('12. Segmentation Overlay (Red)')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print quantitative results
    print("\n=== QUANTITATIVE RESULTS ===")
    print(f"Otsu threshold area: {np.sum(binary_otsu > 0)} pixels")
    print(f"Saliency segmentation area: {np.sum(binary_saliency > 0)} pixels")
    print(f"Combined segmentation area: {np.sum(combined_segmentation > 0)} pixels")
    
    return {
        'original': img_rgb,
        'gray': img_gray,
        'hog_features': hog_features,
        'hog_image': hog_image_rescaled,
        'saliency_map': saliency_map,
        'binary_otsu': binary_otsu,
        'binary_adaptive': binary_adaptive,
        'binary_saliency': binary_saliency,
        'combined_segmentation': combined_segmentation
    }

def compute_saliency_map(gray_image):
    """
    Compute saliency map using spectral residual approach :cite[2]
    """
    # Convert to float32 for processing
    gray_float = np.float32(gray_image)
    
    # Compute Fourier transform
    f = np.fft.fft2(gray_float)
    fshift = np.fft.fftshift(f)
    
    # Compute log amplitude and phase
    magnitude = np.abs(fshift)
    phase = np.angle(fshift)
    
    # Compute spectral residual (simplified approach)
    log_amplitude = np.log(magnitude + 1e-10)  # Avoid log(0)
    
    # Average filter to get background
    avg_kernel = np.ones((3, 3), np.float32) / 9.0
    background = cv2.filter2D(log_amplitude, -1, avg_kernel)
    
    # Spectral residual
    spectral_residual = log_amplitude - background
    
    # Reconstruct saliency map
    saliency = np.abs(np.fft.ifft2(np.exp(spectral_residual + 1j * phase))) ** 2
    
    # Post-processing: Gaussian blur and normalization
    saliency = cv2.GaussianBlur(saliency, (5, 5), 0)
    
    # Normalize to [0, 1]
    saliency = (saliency - np.min(saliency)) / (np.max(saliency) - np.min(saliency) + 1e-10)
    
    return saliency

# Example usage
if __name__ == "__main__":
    # Replace with your TIFF file path
    image_path = "../../data/annotations_uniques/test/images/400-1293-23_3_0_20230307080848.tif"
    
    # Run the complete pipeline
    results = process_image_pipeline(image_path)
    
    if results is not None:
        print("\nPipeline completed successfully!")
        print("Available results: ", list(results.keys()))
    else:
        print("\nPipeline failed! Please check the error messages above.")

In [ ]:
# Load the annotations
input_gt_json_path = '../../data/annotations_uniques/test/COCO_mask/annotations.json'
output_pred_json_path = '../../../Global_Outputs/edt_comparison/fluo_new_edt/predicted_annotations_poly.json'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
from scipy.signal import find_peaks
import os


import json
import matplotlib.pyplot as plt
import tifffile as tiff
from pycocotools import mask as maskUtils
import numpy as np
import cv2
from sklearn import preprocessing as pre


# Introduction to the annotations
 Just checking here if the manual labels and tiff conversion are consistent with each other

In [ ]:
# ---- Paths ----
input_gt_json_path = '../../data/annotations_uniques/test/COCO_mask/annotations.json'
output_pred_json_path = '../../../Global_Outputs/edt_comparison/fluo_new_edt/predicted_annotations_poly.json'
mask_path = "../../data/annotations_uniques/test/masks/400-1605-W8_map_00038_3.tif"

# ---- Load ground truth annotations ----
with open(input_gt_json_path, 'r') as f:
    coco_gt = json.load(f)

image_id_to_name = {img['id']: img['file_name'] for img in coco_gt['images']}
ann_by_image_gt = {}
for ann in coco_gt['annotations']:
    ann_by_image_gt.setdefault(ann['image_id'], []).append(ann)

# ---- Select the image ----
file_name = '400-1605-W8_map_00038_3.jpg'
image_id = [k for k,v in image_id_to_name.items() if v == file_name][0]

# ---- Load tif mask ----
mask_tif = tiff.imread(mask_path)
h, w = mask_tif.shape

# ---- Decode GT mask from annotations.json ----
mask_gt = np.zeros((h, w), dtype=bool)
for ann in ann_by_image_gt[image_id]:
    if len(ann['segmentation']) > 0:
        rle = maskUtils.frPyObjects(ann['segmentation'], h, w)
        decoded = maskUtils.decode(rle)
        if decoded.ndim == 3:  # multiple polygons
            decoded = np.any(decoded, axis=2)
        mask_gt |= decoded.astype(bool)

# ---- Intersection of tif and gt ----
intersection_mask = mask_gt & mask_tif.astype(bool)

# ---- Load predicted annotations ----
with open(output_pred_json_path, 'r') as f:
    coco_pred = json.load(f)

image_id_to_name_pred = {img['id']: img['file_name'] for img in coco_pred['images']}
ann_by_image_pred = {}
for ann in coco_pred['annotations']:
    ann_by_image_pred.setdefault(ann['image_id'], []).append(ann)

# ---- Get predicted mask for same file ----
image_id_pred = [k for k,v in image_id_to_name_pred.items() if v == file_name[:-4]][0]

mask_pred = np.zeros((h, w), dtype=bool)
for ann in ann_by_image_pred[image_id_pred]:
    if len(ann['segmentation']) > 0:
        rle = maskUtils.frPyObjects(ann['segmentation'], h, w)
        decoded = maskUtils.decode(rle)
        if decoded.ndim == 3:
            decoded = np.any(decoded, axis=2)
        mask_pred |= decoded.astype(bool)

# ---- Plotting ----
mismatch = mask_gt ^ mask_tif.astype(bool) 
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(mask_gt, cmap="gray")
axes[0].set_title("GT mask (from annotations.json)")
axes[0].axis("off")

axes[1].imshow(mask_tif, cmap="gray")
axes[1].set_title("TIF mask")
axes[1].axis("off")

axes[2].imshow(mismatch, cmap="gray")
axes[2].set_title("Intersection (GT ∩ TIF)")
axes[2].axis("off")

axes[3].imshow(mask_pred, cmap="gray")
axes[3].set_title("Predicted mask (from predicted_annotations.json)")
axes[3].axis("off")

plt.tight_layout()
plt.show()

gt_missing_in_tif = mask_gt & ~mask_tif.astype(bool)

# Pixels present in tif but missing in GT annotations.json
tif_missing_in_gt = mask_tif.astype(bool) & ~mask_gt
n_gt_missing_in_tif = np.sum(gt_missing_in_tif)
n_tif_missing_in_gt = np.sum(tif_missing_in_gt)

print("Pixels in GT annotations but missing in tif:", n_gt_missing_in_tif)
print("Pixels in tif but missing in GT annotations:", n_tif_missing_in_gt)


# Proof of concept of CND team use-case
(Just a support :) )
## Goal: Compare our orientation/linedensity etraction to their method with FFT

Note: For low density more challenging, for mid and high expected to be more reliable

Method:
* masks as binary because we care about the patterns not their value here
* If low density, tiling the image to increase the periodicity of the patterns but introduces artifacts
* Compute standard magnitude and phase
* Extract Radial profile (for density) ---- because of symmetry no directionality, 0-180


We want to draw correlation between our predicted masks and their FFT feature extraction method.

Still few questions: 
- Do we extract the right density and orientation info from the signal?
- We have line density (count of CNTs per line), this get the density (frequency) of objects in the image at pixel level (objects/pixel).



Remark:
The radial profile is supposed to take into account the symmetry of the objects (see ppt slide 2 in the schematic). So this weights more the majority peak compared to the rest.

In [ ]:

# ---- Load TIFF masks FROM GROUNDTRUTH ----
mask_path = "../../data/annotations_uniques/test/masks"

# mask_path = r"../../../../../data/annotations_filtered_artifacts/test/masks"
mask_filename = "400-1293-18_cd_c1k40r9_fl3_0sp9_0_image_right.tif"
mask_image_path = os.path.join(mask_path, mask_filename)
mask = tiff.imread(mask_image_path)
if mask.ndim == 3 and mask.shape[2] == 3:
    mask = mask.mean(axis=2)
if mask.ndim > 2:
    mask = mask[0]
mask = mask.astype(np.float32)
# we work on binary masks here, we don't care about intensity for orientation and length
# also keeping intensity info cahnges the FFT  results, as not needed we binarize
mask = (mask > 0).astype(np.uint8)

##############################################
#########        ADAPTATION  ############
##############################################
# To work on predicted masks from stardist,
# need to create an array of masks (we loose the intersection info here but we don't care)
# ---- Load mask masks from prediction ----
#
#
#
#
#
#

##############################################
#########        OPTIONAL TILING  ############
##############################################
# FT works with periodicity. One single image has few CNTs so one way is to tile the image 
# to increase the periodicity of the patterns but introduces artifacts 
# so ... Trade off between tiling to get robust results and artifacts

# mask = np.tile(mask, (tile_factor, tile_factor))

# # ---- Optional Hann window ---- 
# filtering method you could try another one if it makes results better

# if apply_window:
#     wy = np.hanning(mask_tiled.shape[0])
#     wx = np.hanning(mask_tiled.shape[1])
#     window = np.outer(wy, wx)
#     mask_tiled *= window


# ---- FFT ----
f = np.fft.fft2(mask)
fshift = np.fft.fftshift(f)
fshift[mask.shape[0]//2, mask.shape[1]//2] = 0
magnitude = np.abs(fshift)

h, w = mask.shape
cy, cx = h//2, w//2

# ---- Polar coordinates ----
Y, X = np.indices((h, w))
R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)
theta = np.arctan2(Y-cy, X-cx)
theta_deg = (np.rad2deg(theta) % 180)  # symmetry

# ---- Radial profile (for density) ----
radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
r = np.arange(len(radial_profile))

peaks, _ = find_peaks(radial_profile, distance=5)
if len(peaks) > 0:
    main_r = peaks[0]
    density_freq = main_r / w  # cycles per pixel
else:
    main_r = None
    density_freq = None

# ---- Angular profile (for orientation) ----
angular_bins = 180
hist, edges = np.histogram(theta_deg, bins=angular_bins, weights=magnitude)
orientation_angle = edges[np.argmax(hist)]
orientation_angle = (orientation_angle + 90) % 180



###################################
# We don't care it's chatGPT
###################################
# ---- Alignment Index ----
alignment_index = np.max(hist) / np.mean(hist)

# ---- Order Parameter ----
if main_r:
    P_peak = radial_profile[main_r]
    mask_bg = (r > main_r + 10) & (r < len(r)//2)
    P_bg = np.mean(radial_profile[mask_bg]) if np.any(mask_bg) else 1
    order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
else:
    order_parameter = None

# ---- Print results ----
print(f"Density frequency: {density_freq:.4f} cycles/pixel" if density_freq else "Density not detected")
print(f"Main orientation: {orientation_angle:.1f}°")
print(f"Alignment Index: {alignment_index:.3f}")
print(f"Order Parameter: {order_parameter:.3f}" if order_parameter is not None else "Order Parameter not detected")

# --- choose the strongest radial peak, not just the first ---
peaks, _ = find_peaks(radial_profile, distance=5)
if len(peaks) > 0:
    main_r = peaks[np.argmax(radial_profile[peaks])]  # <-- strongest peak
else:
    main_r = None

# --- compute density frequency (cycles/pixel) ---
if main_r is not None:
    # NOTE: this normalization assumes a square image; for generality see the “more precise” option below
    density_freq_cyc_per_px = main_r / w
else:
    density_freq_cyc_per_px = None

# --- spacing in pixels and micrometers ---
field_um_x, field_um_y = 5.0, 5.0  # your AFM field size (μm); adjust if different
px_size_um_x = field_um_x / w
px_size_um_y = field_um_y / h
px_size_um = 0.5 * (px_size_um_x + px_size_um_y)  # average (OK for square pixels)

if density_freq_cyc_per_px and density_freq_cyc_per_px > 0:
    spacing_px = 1.0 / density_freq_cyc_per_px               # pixels per period
    spacing_um = spacing_px * px_size_um                      # μm per period
    print(
        f"Dominant spacing: ~{spacing_px:.1f} px  (~{spacing_um:.3f} μm)  "
        f"[pixel size ≈ {px_size_um:.4f} μm/px]"
    )
else:
    print("Dominant spacing: not detected (no clear radial peak)")


# ---- Visualization ----
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Masks
axes[0].imshow(mask, cmap="gray")
axes[0].set_title("FFT Magnitude (log scale)")
axes[0].axis("off")

# FFT magnitude
axes[1].imshow(np.log1p(magnitude), cmap="gray")
axes[1].set_title("FFT Magnitude (log scale)")
axes[1].axis("off")

# Radial profile
axes[2].plot(r, radial_profile, color="black")
if main_r:
    axes[2].axvline(main_r, color="cyan", linestyle="--", label="Main density")
axes[2].set_title("Radial FFT Profile (Density)")
axes[2].set_xlabel("Radius (frequency)")
axes[2].set_ylabel("Energy")
axes[2].legend()

# Angular profile
centers = 0.5 * (edges[:-1] + edges[1:])
axes[3].plot(centers, hist, color="red")
axes[3].axvline(orientation_angle, color="blue", linestyle="--", label="Main orientation")
axes[3].set_title("Angular FFT Profile (Orientation)")
axes[3].set_xlabel("Angle [deg]")
axes[3].set_ylabel("Energy")
axes[3].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ---- Load TIFF masks FROM GROUNDTRUTH ----
mask_path = "../../data/annotations_uniques/test/images"

# mask_path = r"../../../../../daa/annotations_filtered_artifacts/test/masks"
mask_filename = "test.tif"
mask_image_path = os.path.join(mask_path, mask_filename)
mask = tiff.imread(mask_image_path)

# ---- PREPROCESSING PIPELINE ----
def preprocess_mask(mask):
    """Apply preprocessing to enhance FFT results"""
    # Convert to grayscale if needed
    if mask.ndim == 3 and mask.shape[2] == 3:
        mask = mask.mean(axis=2)
    if mask.ndim > 2:
        mask = mask[0]
    
    # Convert to float32 for processing
    mask = mask.astype(np.float32)

    # 1. Normalize intensity to [0, 1] range using MinMaxScaler
    scaler = pre.MinMaxScaler()
    mask_normalized = scaler.fit_transform(mask.reshape(-1, 1)).reshape(mask.shape)

    return mask_normalized

# Example usage:
mask_processed = preprocess_mask(mask)

# ---- FFT ----
mask_processed = mask_processed - np.mean(mask_processed)
f = np.fft.fft2(mask_processed)
fshift = np.fft.fftshift(f)
fshift[mask_processed.shape[0]//2, mask_processed.shape[1]//2] = 0
magnitude = np.abs(fshift)

h, w = mask_processed.shape
cy, cx = h//2, w//2

# ---- FIXED: Polar coordinates with proper Cartesian system ----
Y, X = np.indices((h, w))
# Invert Y axis to match Cartesian coordinates (y increases upward)
Y_cartesian = h - 1 - Y  # This converts image coordinates to Cartesian
R = np.sqrt((X-cx)**2 + (Y_cartesian-cy)**2).astype(int)
theta = np.arctan2(Y_cartesian-cy, X-cx)  # Now this gives proper Cartesian angles
theta_deg = (np.rad2deg(theta) % 180)  # symmetry

# ---- Radial profile (for density) ----
radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
r = np.arange(len(radial_profile))

peaks, _ = find_peaks(radial_profile, distance=5)
if len(peaks) > 0:
    main_r = peaks[0]
    density_freq = main_r / w  # cycles per pixel
else:
    main_r = None
    density_freq = None

# ---- Angular profile (for orientation) ----
angular_bins = 180
hist, edges = np.histogram(theta_deg, bins=angular_bins, weights=magnitude)
orientation_angle = edges[np.argmax(hist)]

# FIXED: No need to add 90 degrees since we're using proper Cartesian coordinates
orientation_angle = (orientation_angle + 90) % 180 

###################################
# We don't care it's chatGPT
###################################
# ---- Alignment Index ----
alignment_index = np.max(hist) / np.mean(hist)

# ---- Order Parameter ----
if main_r:
    P_peak = radial_profile[main_r]
    mask_bg = (r > main_r + 10) & (r < len(r)//2)
    P_bg = np.mean(radial_profile[mask_bg]) if np.any(mask_bg) else 1
    order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
else:
    order_parameter = None

# ---- Print results ----
print(f"Density frequency: {density_freq:.4f} cycles/pixel" if density_freq else "Density not detected")
print(f"Main orientation: {orientation_angle:.1f}°")
print(f"Alignment Index: {alignment_index:.3f}")
print(f"Order Parameter: {order_parameter:.3f}" if order_parameter is not None else "Order Parameter not detected")

# --- choose the strongest radial peak, not just the first ---
peaks, _ = find_peaks(radial_profile, distance=5)
if len(peaks) > 0:
    main_r = peaks[np.argmax(radial_profile[peaks])]  # <-- strongest peak
else:
    main_r = None

# --- compute density frequency (cycles/pixel) ---
field_um_x, field_um_y = 5.0, 5.0  # your AFM field size (μm); adjust if different
px_size_um_x = field_um_x / w
px_size_um_y = field_um_y / h
px_size_um = 0.5 * (px_size_um_x + px_size_um_y)  # average (OK for square pixels)

if main_r and main_r > 0:
    density_freq_cyc_per_px = main_r / w
    spacing_px = 1.0 / density_freq_cyc_per_px               # pixels per period
    spacing_um = spacing_px * px_size_um                      # μm per period
    print(
        f"Dominant spacing: ~{spacing_px:.1f} px  (~{spacing_um:.3f} μm)  "
        f"[pixel size ≈ {px_size_um:.4f} μm/px]"
    )
else:
    print("Dominant spacing: not detected (no clear radial peak)")

# ---- Visualization ----
fig, axes = plt.subplots(1, 6, figsize=(26, 5))

# Original mask
axes[0].imshow(mask, cmap="gray")
axes[0].set_title("Original Mask")
axes[0].axis("off")

# Preprocessed mask
axes[1].imshow(mask_processed)
axes[1].set_title("Preprocessed Mask")
axes[1].axis("off")

# FFT magnitude
axes[2].imshow(np.log1p(magnitude))
axes[2].set_title("FFT Magnitude (log scale)")
axes[2].axis("off")

# Radial profile
axes[3].plot(r, radial_profile, color="black")
if main_r:
    axes[3].axvline(main_r, color="cyan", linestyle="--", label="Main density")
axes[3].set_title("Radial FFT Profile (Density)")
axes[3].set_xlabel("Radius (frequency)")
axes[3].set_ylabel("Energy")
axes[3].legend()

# Angular profile
centers = 0.5 * (edges[:-1] + edges[1:])
axes[4].plot(centers, hist, color="red")
axes[4].axvline(orientation_angle, color="blue", linestyle="--", label="Main orientation")
axes[4].set_title("Angular FFT Profile (Orientation)")
axes[4].set_xlabel("Angle [deg]")
axes[4].set_ylabel("Energy")
axes[4].legend()

# Orientation visualization on mask
axes[5].imshow(mask_processed, cmap="gray")
axes[5].set_title("Orientation Lines")
axes[5].axis("off")

# FIXED: Calculate line endpoints using proper coordinate system conversion
center_y, center_x = h // 2, w // 2
length = min(h, w) * 0.4  # Line length

# Convert angle to radians - this is now in proper Cartesian coordinates
angle_rad = np.radians(orientation_angle)
ortho_angle_rad = angle_rad + np.pi/2  # Orthogonal angle

# FIXED: Convert Cartesian coordinates to image coordinates for plotting
# In image coordinates, y increases downward, so we need to invert the y-component
def cartesian_to_image(x_cart, y_cart, img_height):
    """Convert Cartesian coordinates to image coordinates"""
    return x_cart, img_height - 1 - y_cart

# Calculate endpoints in Cartesian coordinates
end_x_cart = center_x + length * np.cos(angle_rad)
end_y_cart = center_y + length * np.sin(angle_rad)
start_x_cart = center_x - length * np.cos(angle_rad)
start_y_cart = center_y - length * np.sin(angle_rad)

# Calculate orthogonal endpoints in Cartesian coordinates
ortho_end_x_cart = center_x + length * np.cos(ortho_angle_rad)
ortho_end_y_cart = center_y + length * np.sin(ortho_angle_rad)
ortho_start_x_cart = center_x - length * np.cos(ortho_angle_rad)
ortho_start_y_cart = center_y - length * np.sin(ortho_angle_rad)

# Convert all points to image coordinates for plotting
start_x, start_y = cartesian_to_image(start_x_cart, start_y_cart, h)
end_x, end_y = cartesian_to_image(end_x_cart, end_y_cart, h)
ortho_start_x, ortho_start_y = cartesian_to_image(ortho_start_x_cart, ortho_start_y_cart, h)
ortho_end_x, ortho_end_y = cartesian_to_image(ortho_end_x_cart, ortho_end_y_cart, h)

# Plot the lines
axes[5].plot([start_x, end_x], [start_y, end_y], 'r-', linewidth=2, label=f'Main orientation ({orientation_angle:.1f}°)')
axes[5].plot([ortho_start_x, ortho_end_x], [ortho_start_y, ortho_end_y], 'g-', linewidth=2, label='Orthogonal')
axes[5].legend()

plt.tight_layout()
plt.show()

# testing it with fabricated image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# ---- Generate synthetic image with diagonal lines ----
image_size = 256
num_lines = 8
line_width = 5  # thickness of the diagonal lines

# Create blank image
mask = np.zeros((image_size, image_size), dtype=np.uint8)

# Calculate spacing between diagonals
spacing = image_size // num_lines

# Draw diagonal lines at 45° (from bottom-left to top-right)
for i in range(-image_size, image_size, spacing):
    for w in range(-line_width // 2, line_width // 2 + 1):
        # Shifted coordinates for thickness
        y = np.arange(image_size)
        x = y - i - w
        valid = (x >= 0) & (x < image_size)
        mask[y[valid], x[valid].astype(int)] = 1

print(f"Generated image with {num_lines} diagonal lines (45°) and width {line_width}px")
print(f"Line spacing: {spacing} pixels")

# ---- FFT ----
f = np.fft.fft2(mask)
fshift = np.fft.fftshift(f)
fshift[mask.shape[0]//2, mask.shape[1]//2] = 0
magnitude = np.abs(fshift)

h, w = mask.shape
cy, cx = h//2, w//2

# ---- Polar coordinates ----
Y, X = np.indices((h, w))
R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)
theta = np.arctan2(Y-cy, X-cx)
theta_deg = (np.rad2deg(theta) % 180)

# ---- Radial profile ----
radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
r = np.arange(len(radial_profile))

peaks, _ = find_peaks(radial_profile, distance=5)
main_r = peaks[np.argmax(radial_profile[peaks])] if len(peaks) > 0 else None
density_freq = main_r / w if main_r is not None else None

# ---- Angular profile ----
angular_bins = 180
hist, edges = np.histogram(theta_deg, bins=angular_bins, weights=magnitude)
orientation_angle = edges[np.argmax(hist)]
orientation_angle = (orientation_angle + 90) % 180

# ---- Alignment Index ----
alignment_index = np.max(hist) / np.mean(hist)

# ---- Order Parameter ----
if main_r:
    P_peak = radial_profile[main_r]
    mask_bg = (r > main_r + 10) & (r < len(r)//2)
    P_bg = np.mean(radial_profile[mask_bg]) if np.any(mask_bg) else 1
    order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
else:
    order_parameter = None

# ---- Print results ----
print(f"Density frequency: {density_freq:.4f} cycles/pixel" if density_freq else "Density not detected")
print(f"Main orientation: {orientation_angle:.1f}°")
print(f"Alignment Index: {alignment_index:.3f}")
print(f"Order Parameter: {order_parameter:.3f}" if order_parameter is not None else "Order Parameter not detected")

# ---- Visualization ----
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(mask, cmap="gray")
axes[0].set_title(f"Synthetic Image ({num_lines} diagonal lines, 45°, width={line_width}px)")
axes[0].axis("off")

axes[1].imshow(np.log1p(magnitude), cmap="gray")
axes[1].set_title("FFT Magnitude (log scale)")
axes[1].axis("off")

axes[2].plot(r, radial_profile, color="black")
if main_r:
    axes[2].axvline(main_r, color="cyan", linestyle="--", label="Main density")
axes[2].set_title("Radial FFT Profile (Density)")
axes[2].set_xlabel("Radius (frequency)")
axes[2].set_ylabel("Energy")
axes[2].legend()

centers = 0.5 * (edges[:-1] + edges[1:])
axes[3].plot(centers, hist, color="red")
axes[3].axvline(orientation_angle, color="blue", linestyle="--", label="Main orientation")
axes[3].set_title("Angular FFT Profile (Orientation)")
axes[3].set_xlabel("Angle [deg]")
axes[3].set_ylabel("Energy")
axes[3].legend()

plt.tight_layout()
plt.show()


# Extract features from png images

In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile as tiff
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

# Configuration - SET THIS BASED ON WHAT YOU WANT TO PROCESS
PROCESS_PNG = False  # Set to False to process TIFF files
if PROCESS_PNG:
    mask_path = r"C:\Users\abd93000\Desktop\Projects\CNT\inputdata\new_dataset with test\CNT_analysis_tool\cherry_picked_images"
    file_extensions = ('.png',)
else:
    mask_path = "../../data/annotations_uniques/test/images" 
    file_extensions = ('.tif', '.tiff')

def read_image(file_path, is_png):
    """Read image based on file type"""
    if is_png:
        img = cv2.imread(file_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img
    else:
        return tiff.imread(file_path)

def preprocess_mask(mask):
    """Apply preprocessing to enhance FFT results"""
    # Convert to grayscale if needed
    if len(mask.shape) == 3 and mask.shape[2] == 3:
        mask = mask.mean(axis=2)
    if len(mask.shape) > 2:
        mask = mask[0]
    
    # Convert to float32 and normalize
    mask = mask.astype(np.float32)
    scaler = pre.MinMaxScaler()
    return scaler.fit_transform(mask.reshape(-1, 1)).reshape(mask.shape)

def extract_fft_features(mask_processed, field_um_x=5.0, field_um_y=5.0):
    """Extract FFT features from processed mask"""
    try:
        # FFT computation
        mask_processed = mask_processed - np.mean(mask_processed)
        f = np.fft.fft2(mask_processed)
        fshift = np.fft.fftshift(f)
        fshift[mask_processed.shape[0]//2, mask_processed.shape[1]//2] = 0
        magnitude = np.abs(fshift)

        h, w = mask_processed.shape
        cy, cx = h//2, w//2

        # Polar coordinates
        Y, X = np.indices((h, w))
        Y = h - 1 - Y
        R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)
        theta = np.arctan2(Y-cy, X-cx)  
        theta_deg_prof = np.rad2deg(theta)

        # Radial profile
        radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
        r = np.arange(len(radial_profile))

        # Find peaks for density
        peaks, _ = find_peaks(radial_profile, distance=5)
        if len(peaks) > 0:
            main_r = peaks[np.argmax(radial_profile[peaks])]
            density_freq_cyc_per_px = main_r / w
        else:
            main_r = None
            density_freq_cyc_per_px = 0.0

        # Angular profile
        angular_bins = 180
        hist, edges = np.histogram(theta_deg_prof.ravel(), bins=angular_bins, 
                                 range=(0, 180), weights=magnitude.ravel())
        
        fft_orientation_angle = edges[np.argmax(hist)]
        object_orientation_angle = (fft_orientation_angle + 90) % 180

        # Alignment and order parameters
        alignment_index = np.max(hist) / np.mean(hist) if np.mean(hist) > 0 else 1.0
        
        order_parameter = 0.0
        if main_r is not None and main_r < len(radial_profile):
            P_peak = radial_profile[main_r]
            mask_bg = [(r[i] > main_r + 10) and (r[i] < len(r)//2) for i in range(len(r))]
            if any(mask_bg):
                P_bg = np.mean(radial_profile[mask_bg])
                order_parameter = (P_peak - P_bg) / (P_peak + P_bg) if (P_peak + P_bg) > 0 else 0.0

        # Spacing calculation
        px_size_um_x = field_um_x / w
        px_size_um_y = field_um_y / h
        px_size_um = 0.5 * (px_size_um_x + px_size_um_y)

        if density_freq_cyc_per_px > 0:
            spacing_px = 1.0 / density_freq_cyc_per_px
            spacing_um = spacing_px * px_size_um
        else:
            spacing_px = spacing_um = 0.0

        features = {
            'density_freq': float(density_freq_cyc_per_px),
            'fft_orientation_angle': float(fft_orientation_angle),
            'object_orientation_angle': float(object_orientation_angle),
            'alignment_index': float(alignment_index),
            'order_parameter': float(order_parameter),
            'spacing_px': float(spacing_px),
            'spacing_um': float(spacing_um),
            'image_width': int(w),
            'image_height': int(h),
            'px_size_um': float(px_size_um)
        }
        
        return features, True, magnitude, theta_deg_prof, radial_profile, r, hist, edges
        
    except Exception as e:
        print(f"Error in FFT feature extraction: {str(e)}")
        return None, False, None, None, None, None, None, None

def visualize_fft_results(mask, mask_processed, features, magnitude, theta_deg_prof, 
                         radial_profile, r, hist, edges, filename=None):
    """Visualize FFT results showing both FFT and object orientations"""
    h, w = mask_processed.shape
    center_y, center_x = h // 2, w // 2
    
    fig, axes = plt.subplots(1, 6, figsize=(26, 5))
    titles = ["Original Mask", "Preprocessed Mask", "FFT Magnitude (log scale)", 
              "Radial FFT Profile (Density)", "Angular FFT Profile (0°-180°)", 
              "Object Orientation (Actual CNT Direction)"]
    
    # Plot images
    axes[0].imshow(mask, cmap="gray")
    axes[1].imshow(mask_processed)
    axes[2].imshow(np.log1p(magnitude))
    
    # Radial profile
    axes[3].plot(r, radial_profile, color="black")
    if features['density_freq'] > 0:
        main_r = features['density_freq'] * w
        axes[3].axvline(main_r, color="cyan", linestyle="--", label="Main density")
    axes[3].set_xlabel("Radius (frequency)")
    axes[3].set_ylabel("Energy")
    axes[3].legend()

    # Angular profile
    centers = 0.5 * (edges[:-1] + edges[1:])
    axes[4].plot(centers, hist, color="red")
    
    fft_angle = features['fft_orientation_angle']
    object_angle = features['object_orientation_angle']
    
    axes[4].axvline(fft_angle, color="blue", linestyle="--", 
                   label=f'FFT orientation: {fft_angle:.1f}°')
    axes[4].axvline(object_angle, color="green", linestyle="--",
                   label=f'Object orientation: {object_angle:.1f}°')
    axes[4].set_xlabel("Angle [deg] (0°=East, 90°=North, 180°=West)")
    axes[4].set_ylabel("Energy")
    axes[4].set_xlim(0, 180)
    axes[4].legend()

    # Orientation visualization
    axes[5].imshow(mask_processed, cmap="gray")
    length = min(h, w) * 0.4
    object_orientation = features['object_orientation_angle']
    
    # Main orientation line
    angle_rad = np.radians(object_orientation)
    end_x = center_x + length * np.cos(angle_rad)
    end_y = center_y - length * np.sin(angle_rad)
    start_x = center_x - length * np.cos(angle_rad)
    start_y = center_y + length * np.sin(angle_rad)

    # Orthogonal line
    ortho_angle = (object_orientation + 90) % 180
    ortho_angle_rad = np.radians(ortho_angle)
    ortho_end_x = center_x + length * np.cos(ortho_angle_rad)
    ortho_end_y = center_y - length * np.sin(ortho_angle_rad)
    ortho_start_x = center_x - length * np.cos(ortho_angle_rad)
    ortho_start_y = center_y + length * np.sin(ortho_angle_rad)

    axes[5].plot([start_x, end_x], [start_y, end_y], 'r-', linewidth=3, 
                label=f'Object orientation ({object_orientation:.1f}°)')
    axes[5].plot([ortho_start_x, ortho_end_x], [ortho_start_y, ortho_end_y], 'g--', 
                linewidth=2, label=f'Orthogonal (FFT: {features["fft_orientation_angle"]:.1f}°)')
    axes[5].legend()

    # Set titles and turn off axes for image subplots
    for ax, title in zip(axes, titles):
        ax.set_title(title)
        if title not in ["Radial FFT Profile (Density)", "Angular FFT Profile (0°-180°)"]:
            ax.axis("off")

    if filename:
        plt.suptitle(f"FFT Analysis: {filename}", fontsize=14)
    
    plt.tight_layout()
    plt.show()

def create_error_entry(filename, error_msg):
    """Create standardized error entry"""
    return {
        'filename': filename,
        'error': error_msg,
        'density_freq': None, 'fft_orientation_angle': None, 'object_orientation_angle': None,
        'alignment_index': None, 'order_parameter': None, 'spacing_px': None, 'spacing_um': None,
        'image_width': None, 'image_height': None, 'px_size_um': None
    }

def print_statistics(df):
    """Print statistics for successful extractions"""
    successful_df = df[df['error'].isna()]
    if len(successful_df) == 0:
        return
        
    for angle_type in ['object_orientation_angle', 'fft_orientation_angle']:
        valid_angles = successful_df[angle_type].dropna()
        if len(valid_angles) > 0:
            print(f"\n{angle_type.upper()} Statistics (0°-180°):")
            print(f"  Range: {valid_angles.min():.1f}° to {valid_angles.max():.1f}°")
            print(f"  Mean: {valid_angles.mean():.1f}° ± {valid_angles.std():.1f}°")
            print(f"  Median: {valid_angles.median():.1f}°")

# Main processing
results = []
is_png = PROCESS_PNG
file_type = "PNG" if is_png else "TIFF"

# Get files
files = [f for f in os.listdir(mask_path) if f.lower().endswith(file_extensions)]
print(f"Found {len(files)} {file_type} files to process...")

success_count = 0
for i, filename in enumerate(files):
    print(f"Processing {i+1}/{len(files)}: {filename}")
    
    try:
        # Read and preprocess
        file_path = os.path.join(mask_path, filename)
        mask = read_image(file_path, is_png)
        
        if mask is None:
            print(f"  Error: Could not read {filename}")
            results.append(create_error_entry(filename, "Could not read file"))
            continue
            
        mask_processed = preprocess_mask(mask)
        
        # Extract features
        features, success, magnitude, theta_deg_prof, radial_profile, r, hist, edges = extract_fft_features(mask_processed)
        
        if success and features:
            features['filename'] = filename
            features['error'] = None
            results.append(features)
            success_count += 1
            
            print(f"Success! FFT: {features['fft_orientation_angle']:.2f}°, "
                  f"Object: {features['object_orientation_angle']:.2f}°, "
                  f"Alignment: {features['alignment_index']:.2f}, "
                  f"Spacing: {features['spacing_um']:.3f} μm")
            
            # Visualize first few images
            if i < 40:
                visualize_fft_results(mask, mask_processed, features, magnitude, theta_deg_prof,
                                    radial_profile, r, hist, edges, filename)
        else:
            print(f"  Feature extraction failed for {filename}")
            results.append(create_error_entry(filename, 'Feature extraction failed'))
        
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        results.append(create_error_entry(filename, str(e)))

# Create and display results
df = pd.DataFrame(results)
cols = ['filename'] + [col for col in df.columns if col not in ['filename', 'error']] + ['error']
df = df[cols]

print(f"\nProcessing complete! Processed {len(df)} files.")
print(f"Successfully extracted features from {success_count} files.")

# Print statistics
print_statistics(df)

# Display results
print("\nFirst few rows of the DataFrame:")
print(df.head(20))

# Optional: Save to CSV
# output_csv = f"fft_features_results_{'png' if is_png else 'tiff'}_object_orientation.csv"
# df.to_csv(output_csv, index=False)
# print(f"\nResults saved to: {output_csv}")

In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile as tiff
from sklearn import preprocessing as pre
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import cv2

# ---- Load PNG masks ----
mask_path = r"C:\Users\abd93000\Desktop\Projects\CNT\inputdata\new_dataset with test\CNT_analysis_tool\W7\CNT_analysis_tool\images"

def preprocess_mask(mask):
    """Apply preprocessing to enhance FFT results """
    # Convert to grayscale if needed - EXACTLY LIKE YOUR TIFF VERSION
    if len(mask.shape) == 3 and mask.shape[2] == 3:
        mask = mask.mean(axis=2)
    if len(mask.shape) > 2:
        mask = mask[0]
    
    # Convert to float32 for processing
    mask = mask.astype(np.float32)

    # 1. Normalize intensity to [0, 1] range using MinMaxScaler - EXACTLY LIKE YOUR TIFF VERSION
    scaler = pre.MinMaxScaler()
    mask_normalized = scaler.fit_transform(mask.reshape(-1, 1)).reshape(mask.shape)

    return mask_normalized

def extract_fft_features(mask_processed, field_um_x=5.0, field_um_y=5.0):
    """Extract FFT features from processed mask - EXACTLY LIKE YOUR WORKING TIFF VERSION"""
    try:
        # ---- FFT ----
        mask_processed = mask_processed - np.mean(mask_processed)
        f = np.fft.fft2(mask_processed)
        fshift = np.fft.fftshift(f)
        fshift[mask_processed.shape[0]//2, mask_processed.shape[1]//2] = 0
        magnitude = np.abs(fshift)

        h, w = mask_processed.shape
        cy, cx = h//2, w//2

        # ---- Polar coordinates ----
        Y, X = np.indices((h, w))
        Y = h - 1 - Y  # This converts image coordinates to Cartesian
        R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)

        theta = np.arctan2(Y-cy, X-cx)  
        
        # Convert to degrees and get mathematical angle (0°=right, 90°=down)
        theta_deg_prof = np.rad2deg(theta)
        
        # ---- Radial profile (for density) ----
        radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
        r = np.arange(len(radial_profile))

        peaks, _ = find_peaks(radial_profile, distance=5)
        if len(peaks) > 0:
            main_r = peaks[0]
            density_freq = main_r / w  # cycles per pixel
        else:
            main_r = None
            density_freq = None

        # ---- Angular profile (for orientation) ----
        angular_bins = 180
        hist, edges = np.histogram(theta_deg_prof.ravel(), bins=angular_bins, 
                                 range=(0, 180), weights=magnitude.ravel())
                
        fft_orientation_angle = edges[np.argmax(hist)]

        # Convert FFT orientation to actual object orientation
        object_orientation_angle = (fft_orientation_angle + 90) % 180

        # ---- Alignment Index ----
        if np.mean(hist) > 0:
            alignment_index = np.max(hist) / np.mean(hist)
        else:
            alignment_index = 1.0

        # ---- Order Parameter ----
        if main_r is not None and main_r < len(radial_profile):
            P_peak = radial_profile[main_r]
            mask_bg = np.zeros_like(r, dtype=bool)
            for i in range(len(r)):
                if (r[i] > main_r + 10) and (r[i] < len(r)//2):
                    mask_bg[i] = True
            if np.any(mask_bg):
                P_bg = np.mean(radial_profile[mask_bg])
            else:
                P_bg = 1.0
            if (P_peak + P_bg) > 0:
                order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
            else:
                order_parameter = 0.0
        else:
            order_parameter = 0.0

        # --- choose the strongest radial peak, not just the first ---
        peaks, _ = find_peaks(radial_profile, distance=5)
        if len(peaks) > 0:
            main_r = peaks[np.argmax(radial_profile[peaks])]
        else:
            main_r = None

        # --- compute density frequency (cycles/pixel) ---
        if main_r is not None:
            density_freq_cyc_per_px = main_r / w
        else:
            density_freq_cyc_per_px = 0.0

        # --- spacing in pixels and micrometers ---
        px_size_um_x = field_um_x / w
        px_size_um_y = field_um_y / h
        px_size_um = 0.5 * (px_size_um_x + px_size_um_y)

        if density_freq_cyc_per_px and density_freq_cyc_per_px > 0:
            spacing_px = 1.0 / density_freq_cyc_per_px
            spacing_um = spacing_px * px_size_um
        else:
            spacing_px = 0.0
            spacing_um = 0.0

        # Return all features as a dictionary
        features = {
            'density_freq': float(density_freq_cyc_per_px) if density_freq_cyc_per_px else 0.0,
            'fft_orientation_angle': float(fft_orientation_angle),
            'object_orientation_angle': float(object_orientation_angle),
            'alignment_index': float(alignment_index),
            'order_parameter': float(order_parameter),
            'spacing_px': float(spacing_px),
            'spacing_um': float(spacing_um),
            'image_width': int(w),
            'image_height': int(h),
            'px_size_um': float(px_size_um)
        }
        
        return features, True, magnitude, theta_deg_prof, radial_profile, r, hist, edges
        
    except Exception as e:
        print(f"Error in FFT feature extraction: {str(e)}")
        return None, False, None, None, None, None, None, None

def visualize_fft_results(mask, mask_processed, features, magnitude, theta_deg_prof, 
                         radial_profile, r, hist, edges, filename=None):
    """Visualize FFT results showing both FFT and object orientations"""
    h, w = mask_processed.shape
    center_y, center_x = h // 2, w // 2
    
    fig, axes = plt.subplots(1, 6, figsize=(26, 5))

    # Original mask
    axes[0].imshow(mask, cmap="gray")
    axes[0].set_title("Original Mask")
    axes[0].axis("off")

    # Preprocessed mask
    axes[1].imshow(mask_processed)
    axes[1].set_title("Preprocessed Mask")
    axes[1].axis("off")

    # FFT magnitude
    axes[2].imshow(np.log1p(magnitude))
    axes[2].set_title("FFT Magnitude (log scale)")
    axes[2].axis("off")

    # Radial profile
    axes[3].plot(r, radial_profile, color="black")
    if features['density_freq'] is not None and features['density_freq'] > 0:
        main_r = features['density_freq'] * w
        axes[3].axvline(main_r, color="cyan", linestyle="--", label="Main density")
    axes[3].set_title("Radial FFT Profile (Density)")
    axes[3].set_xlabel("Radius (frequency)")
    axes[3].set_ylabel("Energy")
    axes[3].legend()

    # Angular profile (professional coordinates 0-180°)
    centers = 0.5 * (edges[:-1] + edges[1:])
    axes[4].plot(centers, hist, color="red")
    
    # Show both FFT and object orientations
    fft_angle = features['fft_orientation_angle']
    object_angle = features['object_orientation_angle']
    
    axes[4].axvline(fft_angle, color="blue", linestyle="--", 
                   label=f'FFT orientation: {fft_angle:.1f}°')
    axes[4].axvline(object_angle, color="green", linestyle="--",
                   label=f'Object orientation: {object_angle:.1f}°')
    
    axes[4].set_title("Angular FFT Profile (0°-180°)")
    axes[4].set_xlabel("Angle [deg] (0°=East, 90°=North, 180°=West)")
    axes[4].set_ylabel("Energy")
    axes[4].set_xlim(0, 180)
    axes[4].legend()

    # Orientation visualization on mask - show OBJECT orientation
    axes[5].imshow(mask_processed, cmap="gray")
    axes[5].set_title("Object Orientation (Actual CNT Direction)")
    axes[5].axis("off")

    # Calculate line endpoints using OBJECT orientation (not FFT orientation)
    object_orientation = features['object_orientation_angle']
    length = min(h, w) * 0.4

    angle_rad = np.radians(object_orientation)
    
    # Calculate endpoints for main object orientation line (red)
    end_x = center_x + length * np.cos(angle_rad)
    end_y = center_y - length * np.sin(angle_rad)
    start_x = center_x - length * np.cos(angle_rad)
    start_y = center_y + length * np.sin(angle_rad)

    # Calculate endpoints for orthogonal line (green)
    ortho_angle = (object_orientation + 90) % 180
    ortho_angle_rad = np.radians(ortho_angle)
    ortho_end_x = center_x + length * np.cos(ortho_angle_rad)
    ortho_end_y = center_y - length * np.sin(ortho_angle_rad)
    ortho_start_x = center_x - length * np.cos(ortho_angle_rad)
    ortho_start_y = center_y + length * np.sin(ortho_angle_rad)

    # Plot the lines
    axes[5].plot([start_x, end_x], [start_y, end_y], 'r-', linewidth=3, 
                label=f'Object orientation ({object_orientation:.1f}°)')
    axes[5].plot([ortho_start_x, ortho_end_x], [ortho_start_y, ortho_end_y], 'g--', 
                linewidth=2, label=f'Orthogonal (FFT: {features["fft_orientation_angle"]:.1f}°)')
    axes[5].legend()

    if filename:
        plt.suptitle(f"FFT Analysis: {filename}", fontsize=14)
    
    plt.tight_layout()
    plt.show()

# Initialize DataFrame to store results
results = []

# Get all PNG files in the folder
png_files = [f for f in os.listdir(mask_path) if f.endswith('.png')]

print(f"Found {len(png_files)} PNG files to process...")
print("NOTE: Object orientation = FFT orientation + 90° (orthogonal direction)")

# Process each image
success_count = 0
for i, mask_filename in enumerate(png_files):
    print(f"Processing {i+1}/{len(png_files)}: {mask_filename}")
    
    try:
        # Read and preprocess mask - CRITICAL: Use the same preprocessing as TIFF
        mask_image_path = os.path.join(mask_path, mask_filename)
        
        # Read PNG with OpenCV but convert to match TIFF format
        mask_cv = cv2.imread(mask_image_path)
        
        # Convert BGR to RGB and ensure same processing as TIFF
        if mask_cv is not None:
            # Convert BGR to RGB
            mask_rgb = cv2.cvtColor(mask_cv, cv2.COLOR_BGR2RGB)
            
            # If the image has 3 channels, convert to grayscale using mean (like your TIFF code)
            if len(mask_rgb.shape) == 3 and mask_rgb.shape[2] == 3:
                mask = mask_rgb.mean(axis=2)
            else:
                mask = mask_rgb
                
            print(f"  Image shape: {mask.shape}, range: [{np.min(mask):.1f}, {np.max(mask):.1f}]")
        else:
            print(f"  Error: Could not read {mask_filename}")
            continue
            
        mask_processed = preprocess_mask(mask)
        
        # Extract features
        features, success, magnitude, theta_deg_prof, radial_profile, r, hist, edges = extract_fft_features(mask_processed)
        
        if success and features:
            features['filename'] = mask_filename
            features['error'] = None
            results.append(features)
            success_count += 1
            
            # Print feature summary
            print(f"Success! FFT orientation: {features['fft_orientation_angle']:.2f}°, "
                  f"Object orientation: {features['object_orientation_angle']:.2f}°, "
                  f"alignment_index: {features['alignment_index']:.2f}, "
                  f"Spacing: {features['spacing_um']:.3f} μm")
            
            # Visualize first few images
            if i < 40:
                visualize_fft_results(mask, mask_processed, features, magnitude, theta_deg_prof,
                                    radial_profile, r, hist, edges, mask_filename)
        else:
            print(f"  Feature extraction failed for {mask_filename}")
            results.append({
                'filename': mask_filename,
                'error': 'Feature extraction failed',
                'density_freq': None,
                'fft_orientation_angle': None,
                'object_orientation_angle': None,
                'alignment_index': None,
                'order_parameter': None,
                'spacing_px': None,
                'spacing_um': None,
                'image_width': None,
                'image_height': None,
                'px_size_um': None
            })
        
    except Exception as e:
        print(f"Error processing {mask_filename}: {str(e)}")
        import traceback
        traceback.print_exc()
        results.append({
            'filename': mask_filename,
            'error': str(e),
            'density_freq': None,
            'fft_orientation_angle': None,
            'object_orientation_angle': None,
            'alignment_index': None,
            'order_parameter': None,
            'spacing_px': None,
            'spacing_um': None,
            'image_width': None,
            'image_height': None,
            'px_size_um': None
        })

# Create DataFrame
df = pd.DataFrame(results)

# Reorder columns to have filename first
cols = ['filename'] + [col for col in df.columns if col != 'filename' and col != 'error'] + ['error']
df = df[cols]

# Display summary
print(f"\nProcessing complete! Processed {len(df)} files.")
print(f"Successfully extracted features from {success_count} files.")

# Display orientation statistics for successful extractions
successful_df = df[df['error'].isna()]
if len(successful_df) > 0:
    valid_object_orientations = successful_df['object_orientation_angle'].dropna()
    valid_fft_orientations = successful_df['fft_orientation_angle'].dropna()
    
    if len(valid_object_orientations) > 0:
        print(f"\nOBJECT ORIENTATION Statistics (0°-180°):")
        print(f"  Range: {valid_object_orientations.min():.1f}° to {valid_object_orientations.max():.1f}°")
        print(f"  Mean: {valid_object_orientations.mean():.1f}° ± {valid_object_orientations.std():.1f}°")
        print(f"  Median: {valid_object_orientations.median():.1f}°")
        
        print(f"\nFFT ORIENTATION Statistics (0°-180°):")
        print(f"  Range: {valid_fft_orientations.min():.1f}° to {valid_fft_orientations.max():.1f}°")
        print(f"  Mean: {valid_fft_orientations.mean():.1f}° ± {valid_fft_orientations.std():.1f}°")
        print(f"  Median: {valid_fft_orientations.median():.1f}°")
        
        # Verify orthogonal relationship
        diff_angles = (valid_object_orientations.values - valid_fft_orientations.values) % 180
        print(f"\nOrthogonality check (Object - FFT) % 180:")
        print(f"  Mean difference: {np.mean(diff_angles):.1f}° (should be close to 90°)")
        print(f"  Std of difference: {np.std(diff_angles):.1f}°")

# Display first few rows
print("\nFirst few rows of the DataFrame:")
print(df.head(20))

# Save to CSV
# output_csv = "fft_features_results_png_object_orientation.csv"
# df.to_csv(output_csv, index=False)
# print(f"\nResults saved to: {output_csv}")

# Optional: Display summary statistics for numerical columns
numerical_cols = ['density_freq', 'fft_orientation_angle', 'object_orientation_angle', 
                  'alignment_index', 'order_parameter', 'spacing_px', 'spacing_um']
available_numerical_cols = [col for col in numerical_cols if col in df.columns]
if available_numerical_cols:
    print("\nSummary statistics:")
    print(df[available_numerical_cols].describe())

# Compare prediction based features with fft based features: TIF Files

In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile as tiff
from sklearn import preprocessing as pre
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
import cv2

# ---- Load TIFF masks FROM GROUNDTRUTH ----
mask_path = "../../data/annotations_uniques/test/images"
# mask_path = r"C:\Users\abd93000\Desktop\Projects\CNT\inputdata\new_dataset with test\CNT_analysis_tool\W7\CNT_analysis_tool\images"


def preprocess_mask(mask):
    """Apply preprocessing to enhance FFT results"""
    # Convert to grayscale if needed
    if len(mask.shape) == 3 and mask.shape[2] == 3:
        mask = mask.mean(axis=2)
    if len(mask.shape) > 2:
        mask = mask[0]
    
    # Convert to float32 for processing
    mask = mask.astype(np.float32)

    # 1. Normalize intensity to [0, 1] range using MinMaxScaler
    scaler = pre.MinMaxScaler()
    mask_normalized = scaler.fit_transform(mask.reshape(-1, 1)).reshape(mask.shape)

    return mask_normalized


def extract_fft_features(mask_processed, field_um_x=5.0, field_um_y=5.0):
    """Extract FFT features from processed mask"""
    try:
        # ---- FFT ----
        mask_processed = mask_processed - np.mean(mask_processed)
        f = np.fft.fft2(mask_processed)
        fshift = np.fft.fftshift(f)
        # center_y, center_x = mask_processed.shape[0]//2, mask_processed.shape[1]//2
        # fshift[center_y, center_x] = 0
        fshift[mask_processed.shape[0]//2, mask_processed.shape[1]//2] = 0
        magnitude = np.abs(fshift)

        h, w = mask_processed.shape
        cy, cx = h//2, w//2

        # ---- Polar coordinates ----
        Y, X = np.indices((h, w))
        Y = h - 1 - Y  # This converts image coordinates to Cartesian
        # R = np.sqrt((X-center_x)**2 + (Y-center_y)**2).astype(int)
        R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)

        theta = np.arctan2(Y-cy, X-cx)  

        
        # Convert to degrees and get mathematical angle (0°=right, 90°=down)
        theta_deg_prof = np.rad2deg(theta)
        

        # ---- Radial profile (for density) ----
        radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
        r = np.arange(len(radial_profile))

        peaks, _ = find_peaks(radial_profile, distance=5)
        if len(peaks) > 0:
            main_r = peaks[0]
            density_freq = main_r / w  # cycles per pixel
        else:
            main_r = None
            density_freq = None

        # ---- Angular profile (for orientation) ----
        # Use professional coordinate system for angular analysis (0-180° range)
        angular_bins = 180 # 180
        # Use the professional coordinates for histogram with 0-180° range
        hist, edges = np.histogram(theta_deg_prof.ravel(), bins=angular_bins, 
                                 range=(0, 180), weights=magnitude.ravel())
                
        # fft_orientation_angle = edges[np.argmax(hist)]
        # Use bin centers instead of edges for better accuracy
        fft_orientation_angle = edges[np.argmax(hist)]

        # Convert FFT orientation to actual object orientation
        object_orientation_angle = (fft_orientation_angle + 90) % 180


        # ---- Alignment Index ----
        if np.mean(hist) > 0:
            alignment_index = np.max(hist) / np.mean(hist)
        else:
            alignment_index = 1.0

        # ---- Order Parameter ----
        if main_r is not None and main_r < len(radial_profile):
            P_peak = radial_profile[main_r]
            # Create boolean mask safely
            mask_bg = np.zeros_like(r, dtype=bool)
            for i in range(len(r)):
                if (r[i] > main_r + 10) and (r[i] < len(r)//2):
                    mask_bg[i] = True
            if np.any(mask_bg):
                P_bg = np.mean(radial_profile[mask_bg])
            else:
                P_bg = 1.0
            if (P_peak + P_bg) > 0:
                order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
            else:
                order_parameter = 0.0
        else:
            order_parameter = 0.0

        # --- choose the strongest radial peak, not just the first ---
        peaks, _ = find_peaks(radial_profile, distance=5)
        if len(peaks) > 0:
            main_r = peaks[np.argmax(radial_profile[peaks])]  # <-- strongest peak
        else:
            main_r = None

        # --- compute density frequency (cycles/pixel) ---
        if main_r is not None:
            density_freq_cyc_per_px = main_r / w
        else:
            density_freq_cyc_per_px = 0.0

        # --- spacing in pixels and micrometers ---
        px_size_um_x = field_um_x / w
        px_size_um_y = field_um_y / h
        px_size_um = 0.5 * (px_size_um_x + px_size_um_y)  # average (OK for square pixels)

        if density_freq_cyc_per_px and density_freq_cyc_per_px > 0:
            spacing_px = 1.0 / density_freq_cyc_per_px  # pixels per period
            spacing_um = spacing_px * px_size_um        # μm per period
        else:
            spacing_px = 0.0
            spacing_um = 0.0

        # Return all features as a dictionary
        features = {
            'density_freq': float(density_freq_cyc_per_px) if density_freq_cyc_per_px else 0.0,
            'fft_orientation_angle': float(fft_orientation_angle),  # Keep FFT orientation for reference
            'object_orientation_angle': float(object_orientation_angle),  # Actual object orientation
            'alignment_index': float(alignment_index),
            'order_parameter': float(order_parameter),
            'spacing_px': float(spacing_px),
            'spacing_um': float(spacing_um),
            'image_width': int(w),
            'image_height': int(h),
            'px_size_um': float(px_size_um)
        }
        
        return features, True, magnitude, theta_deg_prof, radial_profile, r, hist, edges
        
    except Exception as e:
        print(f"Error in FFT feature extraction: {str(e)}")
        return None, False, None, None, None, None, None, None

def visualize_fft_results(mask, mask_processed, features, magnitude, theta_deg_prof, 
                         radial_profile, r, hist, edges, filename=None):
    """Visualize FFT results showing both FFT and object orientations"""
    h, w = mask_processed.shape
    center_y, center_x = h // 2, w // 2
    
    fig, axes = plt.subplots(1, 6, figsize=(26, 5))

    # Original mask
    axes[0].imshow(mask, cmap="gray")
    axes[0].set_title("Original Mask")
    axes[0].axis("off")

    # Preprocessed mask
    axes[1].imshow(mask_processed)
    axes[1].set_title("Preprocessed Mask")
    axes[1].axis("off")

    # FFT magnitude
    axes[2].imshow(np.log1p(magnitude))
    axes[2].set_title("FFT Magnitude (log scale)")
    axes[2].axis("off")

    # Radial profile
    axes[3].plot(r, radial_profile, color="black")
    if features['density_freq'] is not None and features['density_freq'] > 0:
        main_r = features['density_freq'] * w
        axes[3].axvline(main_r, color="cyan", linestyle="--", label="Main density")
    axes[3].set_title("Radial FFT Profile (Density)")
    axes[3].set_xlabel("Radius (frequency)")
    axes[3].set_ylabel("Energy")
    axes[3].legend()

    # Angular profile (professional coordinates 0-180°)
    centers = 0.5 * (edges[:-1] + edges[1:])
    axes[4].plot(centers, hist, color="red")
    
    # Show both FFT and object orientations
    fft_angle = features['fft_orientation_angle']
    object_angle = features['object_orientation_angle']
    
    axes[4].axvline(fft_angle, color="blue", linestyle="--", 
                   label=f'FFT orientation: {fft_angle:.1f}°')
    axes[4].axvline(object_angle, color="green", linestyle="--",
                   label=f'Object orientation: {object_angle:.1f}°')
    
    axes[4].set_title("Angular FFT Profile (0°-180°)")
    axes[4].set_xlabel("Angle [deg] (0°=East, 90°=North, 180°=West)")
    axes[4].set_ylabel("Energy")
    axes[4].set_xlim(0, 180)
    axes[4].legend()

    # Orientation visualization on mask - show OBJECT orientation
    axes[5].imshow(mask_processed, cmap="gray")
    axes[5].set_title("Object Orientation (Actual CNT Direction)")
    axes[5].axis("off")

    # Calculate line endpoints using OBJECT orientation (not FFT orientation)
    object_orientation = features['object_orientation_angle']
    length = min(h, w) * 0.4  # Line length

    # Convert object orientation to radians
    # In professional coords: 0°=right, 90°=up, 180°=left
    angle_rad = np.radians(object_orientation)
    
    # Calculate endpoints for main object orientation line (red)
    end_x = center_x + length * np.cos(angle_rad)
    end_y = center_y - length * np.sin(angle_rad)  # Subtract because image y points down
    start_x = center_x - length * np.cos(angle_rad)
    start_y = center_y + length * np.sin(angle_rad)  # Add because image y points down

    # Calculate endpoints for orthogonal line (green) - this should match FFT orientation
    ortho_angle = (object_orientation + 90) % 180  # Orthogonal angle in 0-180° range
    ortho_angle_rad = np.radians(ortho_angle)
    ortho_end_x = center_x + length * np.cos(ortho_angle_rad)
    ortho_end_y = center_y - length * np.sin(ortho_angle_rad)
    ortho_start_x = center_x - length * np.cos(ortho_angle_rad)
    ortho_start_y = center_y + length * np.sin(ortho_angle_rad)

    # Plot the lines
    axes[5].plot([start_x, end_x], [start_y, end_y], 'r-', linewidth=3, 
                label=f'Object orientation ({object_orientation:.1f}°)')
    axes[5].plot([ortho_start_x, ortho_end_x], [ortho_start_y, ortho_end_y], 'g--', 
                linewidth=2, label=f'Orthogonal (FFT: {features["fft_orientation_angle"]:.1f}°)')
    axes[5].legend()

    if filename:
        plt.suptitle(f"FFT Analysis: {filename}", fontsize=14)
    
    plt.tight_layout()
    plt.show()

# Initialize DataFrame to store results
results = []

# Get all TIFF files in the folder
tiff_files = [f for f in os.listdir(mask_path) if f.endswith(('.tif', '.tiff'))]

print(f"Found {len(tiff_files)} TIFF files to process...")
print("NOTE: Object orientation = FFT orientation + 90° (orthogonal direction)")

# Process each image
success_count = 0
for i, mask_filename in enumerate(tiff_files):
    print(f"Processing {i+1}/{len(tiff_files)}: {mask_filename}")
    
    try:
        # Read and preprocess mask
        mask_image_path = os.path.join(mask_path, mask_filename)
        mask = tiff.imread(mask_image_path)
        mask_processed = preprocess_mask(mask)
        
        # Extract features
        features, success, magnitude, theta_deg_prof, radial_profile, r, hist, edges = extract_fft_features(mask_processed)
        
        if success and features:
            # Add filename to features
            features['filename'] = mask_filename
            features['error'] = None
            
            # Append to results
            results.append(features)
            success_count += 1
            
            # Print feature summary for first few images
            if i < 43:  # Visualize first 3 images
                print(f"  Success! FFT orientation: {features['fft_orientation_angle']:.2f}°, "
                      f"Object orientation: {features['object_orientation_angle']:.2f}°, "
                      f"alignment_index: {features['alignment_index']:.2f}, "
                      f"Spacing: {features['spacing_um']:.3f} μm")
                visualize_fft_results(mask, mask_processed, features, magnitude, theta_deg_prof,
                                    radial_profile, r, hist, edges, mask_filename)
            else:
                print(f"  Success! FFT: {features['fft_orientation_angle']:.2f}°, "
                      f"Object: {features['object_orientation_angle']:.2f}°, "
                        f"alignment_index: {features['alignment_index']:.2f}, "
                      f"Spacing: {features['spacing_um']:.3f} μm")
        else:
            print(f"  Feature extraction failed for {mask_filename}")
            results.append({
                'filename': mask_filename,
                'error': 'Feature extraction failed',
                'density_freq': None,
                'fft_orientation_angle': None,
                'object_orientation_angle': None,
                'alignment_index': None,
                'order_parameter': None,
                'spacing_px': None,
                'spacing_um': None,
                'image_width': None,
                'image_height': None,
                'px_size_um': None
            })
        
    except Exception as e:
        print(f"Error processing {mask_filename}: {str(e)}")
        # Add error entry
        results.append({
            'filename': mask_filename,
            'error': str(e),
            'density_freq': None,
            'fft_orientation_angle': None,
            'object_orientation_angle': None,
            'alignment_index': None,
            'order_parameter': None,
            'spacing_px': None,
            'spacing_um': None,
            'image_width': None,
            'image_height': None,
            'px_size_um': None
        })

# Create DataFrame
df = pd.DataFrame(results)

# Reorder columns to have filename first
cols = ['filename'] + [col for col in df.columns if col != 'filename' and col != 'error'] + ['error']
df = df[cols]

# Display summary
print(f"\nProcessing complete! Processed {len(df)} files.")
print(f"Successfully extracted features from {success_count} files.")

# Display orientation statistics for successful extractions
successful_df = df[df['error'].isna()]
if len(successful_df) > 0:
    valid_object_orientations = successful_df['object_orientation_angle'].dropna()
    valid_fft_orientations = successful_df['fft_orientation_angle'].dropna()
    
    if len(valid_object_orientations) > 0:
        print(f"\nOBJECT ORIENTATION Statistics (0°-180°):")
        print(f"  Range: {valid_object_orientations.min():.1f}° to {valid_object_orientations.max():.1f}°")
        print(f"  Mean: {valid_object_orientations.mean():.1f}° ± {valid_object_orientations.std():.1f}°")
        print(f"  Median: {valid_object_orientations.median():.1f}°")
        
        print(f"\nFFT ORIENTATION Statistics (0°-180°):")
        print(f"  Range: {valid_fft_orientations.min():.1f}° to {valid_fft_orientations.max():.1f}°")
        print(f"  Mean: {valid_fft_orientations.mean():.1f}° ± {valid_fft_orientations.std():.1f}°")
        print(f"  Median: {valid_fft_orientations.median():.1f}°")
        
        # Verify orthogonal relationship
        diff_angles = (valid_object_orientations.values - valid_fft_orientations.values) % 180
        print(f"\nOrthogonality check (Object - FFT) % 180:")
        print(f"  Mean difference: {np.mean(diff_angles):.1f}° (should be close to 90°)")
        print(f"  Std of difference: {np.std(diff_angles):.1f}°")

# Display first few rows
print("\nFirst few rows of the DataFrame:")
print(df.head(20))

# Save to CSV
output_csv = "fft_features_results_object_orientation.csv"
df.to_csv(output_csv, index=False)
print(f"\nResults saved to: {output_csv}")

# Optional: Display summary statistics for numerical columns
numerical_cols = ['density_freq', 'fft_orientation_angle', 'object_orientation_angle', 
                  'alignment_index', 'order_parameter', 'spacing_px', 'spacing_um']
available_numerical_cols = [col for col in numerical_cols if col in df.columns]
if available_numerical_cols:
    print("\nSummary statistics:")
    print(df[available_numerical_cols].describe())

In [ ]:
import pandas as pd
import numpy as np

# Read the CSV file
csv_df = pd.read_csv(r'C:\Users\abd93000\Desktop\Projects\CNT\inputdata\new_dataset with test\CNT_analysis_tool\W7\CNT_analysis_tool\image_level_averages.csv')  # Replace with your actual CSV file path

# Change .png extension to .tif in the filename column
# csv_df['filename'] = csv_df['filename'].str.replace('.png', '.tif', regex=False)

# Convert orientation angles from [-90, 90] to [0, 180]
def convert_orientation(angle):
    """
    Convert orientation angle from [-90, 90] range to [0, 180] range
    """
    if pd.isna(angle):
        return angle
    # Add 180 to negative angles, keep positive angles as is
    if angle < 0:
        return angle + 180
    else:
        return angle

# Apply conversion to both orientation columns
csv_df['von_mises_mean_orientation_deg'] = csv_df['von_mises_mean_orientation_deg'].apply(convert_orientation)
csv_df['avg_orientation_angle_deg'] = csv_df['avg_orientation_angle_deg'].apply(convert_orientation)



# Correlate Everything

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Merge DataFrames on 'filename'
merged_df = pd.merge(df, csv_df, on='filename', how='inner')

# Select only numerical columns
numerical_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude specified columns
exclude_cols = ['image_width', 'image_height', 'px_size_um']
exclude_cols += [col for col in numerical_cols if 'std' in col.lower()]

# Filter the columns
numerical_cols = [col for col in numerical_cols if col not in exclude_cols]


print(f"Columns included in correlation analysis ({len(numerical_cols)}):")
for col in numerical_cols:
    print(f"  - {col}")

# Create the numerical dataframe
numerical_df = merged_df[numerical_cols].copy()


# Initialize correlation matrices with proper NaN values
corr_matrix = pd.DataFrame(np.nan, index=numerical_cols, columns=numerical_cols)
p_value_matrix = pd.DataFrame(np.nan, index=numerical_cols, columns=numerical_cols)

# Calculate correlations using a more robust approach
for i, col1 in enumerate(numerical_cols):
    for j, col2 in enumerate(numerical_cols):
        if i <= j:  # Calculate only upper triangle to avoid redundancy
            # Get the data for these two columns
            data1 = numerical_df[col1].values
            data2 = numerical_df[col2].values
            
            # Create a mask for non-NaN values in both columns
            mask = ~(np.isnan(data1) | np.isnan(data2))
            
            if np.sum(mask) >= 3:  # Need at least 3 data points
                try:
                    # Extract clean data
                    clean_data1 = data1[mask]
                    clean_data2 = data2[mask]
                    
                    # Ensure we have 1D arrays
                    clean_data1 = np.asarray(clean_data1).flatten()
                    clean_data2 = np.asarray(clean_data2).flatten()
                    
                    # Calculate correlation
                    corr, p_value = pearsonr(clean_data1, clean_data2)
                    
                    # Store results
                    corr_matrix.loc[col1, col2] = corr
                    corr_matrix.loc[col2, col1] = corr
                    p_value_matrix.loc[col1, col2] = p_value
                    p_value_matrix.loc[col2, col1] = p_value
                    
                except Exception as e:
                    print(f"Error calculating {col1} vs {col2}: {str(e)[:100]}...")
                    continue

print("\nCorrelation Matrix:")
# Display with better formatting
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(corr_matrix.round(4))

print("\nP-Value Matrix:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(p_value_matrix.round(6))

# Find and display significant correlations
print("\n" + "="*80)
print("SIGNIFICANT CORRELATIONS (p < 0.05)")
print("="*80)

significant_correlations = []
for i, col1 in enumerate(numerical_cols):
    for j, col2 in enumerate(numerical_cols):
        if i < j:  # Only upper triangle to avoid duplicates
            corr = corr_matrix.loc[col1, col2]
            p_val = p_value_matrix.loc[col1, col2]
            
            if not np.isnan(corr) and not np.isnan(p_val):
                if p_val < 0.05:
                    significant_correlations.append({
                        'Variable 1': col1,
                        'Variable 2': col2,
                        'Correlation': round(corr, 4),
                        'P-Value': f"{p_val:.6f}",
                        'Strength': 'Strong' if abs(corr) > 0.7 else 
                                   'Moderate' if abs(corr) > 0.5 else 
                                   'Weak' if abs(corr) > 0.3 else 'Very Weak'
                    })

if significant_correlations:
    sig_df = pd.DataFrame(significant_correlations)
    # Sort by absolute correlation value (strongest first)
    sig_df['abs_corr'] = sig_df['Correlation'].abs()
    sig_df = sig_df.sort_values('abs_corr', ascending=False).drop('abs_corr', axis=1)
    
    print(f"Found {len(sig_df)} significant correlations:")
    print(sig_df.to_string(index=False))
    
    # Also show summary by strength
    print("\nSummary by strength:")
    strength_summary = sig_df['Strength'].value_counts()
    for strength, count in strength_summary.items():
        print(f"  {strength}: {count}")
else:
    print("No significant correlations found (p < 0.05)")

# Optional: Create a simplified correlation matrix showing only significant correlations
print("\n" + "="*80)
print("SIMPLIFIED CORRELATION MATRIX (Only showing |r| > 0.3)")
print("="*80)

# Create a simplified version for better readability
simplified_corr = corr_matrix.copy()
for col1 in numerical_cols:
    for col2 in numerical_cols:
        corr_val = simplified_corr.loc[col1, col2]
        p_val = p_value_matrix.loc[col1, col2]
        
        if np.isnan(corr_val) or abs(corr_val) < 0.3:
            simplified_corr.loc[col1, col2] = np.nan
        else:
            # Format as r-value with significance stars
            stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
            simplified_corr.loc[col1, col2] = f"{corr_val:.3f}{stars}"

# Remove rows and columns with all NaN values
simplified_corr = simplified_corr.dropna(axis=0, how='all').dropna(axis=1, how='all')

if not simplified_corr.empty:
    print("Correlations with |r| > 0.3:")
    print(simplified_corr)
else:
    print("No correlations with |r| > 0.3 found")


import seaborn as sns
import numpy as np
import matplotlib

# Use a monospaced font for clean stacking
matplotlib.rcParams['font.family'] = 'DejaVu Sans Mono'

# --- Build combined annotation text (r + p) ---
annot_text = corr_matrix.copy().astype(str)

for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        p = p_value_matrix.iloc[i, j]

        if not np.isnan(r) and not np.isnan(p):
            # add spacing line between correlation and p
            annot_text.iloc[i, j] = f"{r:.2f}\np:{p:.3f}"
        else:
            annot_text.iloc[i, j] = ""

# --- Dynamic figure sizing with more generous scaling ---
num_cols = len(corr_matrix.columns)
cell_size = 0.9  # controls box size per variable
fig_width = max(14, num_cols * cell_size)
fig_height = max(12, num_cols * cell_size)

plt.figure(figsize=(fig_width, fig_height))

# --- Plot heatmap ---
sns.heatmap(
    corr_matrix.astype(float),
    annot=annot_text,
    fmt="",
    cmap="coolwarm",
    center=0,
    linewidths=0.4,
    cbar_kws={"shrink": 0.6, "label": "Pearson correlation"},
    annot_kws={
        "size": 9,            # slightly larger font
        "va": "center",       # vertically centered
        "ha": "center",       # horizontally centered
        "linespacing": 1.6    # extra space between correlation & p-value
    }
)

plt.title("Correlation Matrix with P-Values", fontsize=18, pad=24)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Merge dataframes and start analyzing

In [ ]:
import pandas as pd

# ---- Merge both DataFrames on 'filename' ----
merged = pd.merge(
    df[['filename', 'object_orientation_angle']],
    csv_df[['filename', 'von_mises_mean_orientation_deg', 'avg_orientation_angle_deg', "von_mises_concentration_kappa"]],
    on='filename',
    how='inner'
)

# ---- Sort for cleaner display ----
merged = merged.sort_values(by='filename').reset_index(drop=True)

# ---- Style the table ----
styled_table = (
    merged.style
    .set_caption("Orientation Comparison Table")
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '16px'),
            ('font-weight', 'bold'),
            ('text-align', 'center'),
            ('color', '#222')
        ]},
        {'selector': 'th', 'props': [
            ('background-color', '#f0f0f0'),
            ('color', '#222'),
            ('font-size', '13px'),
            ('font-weight', 'bold'),
            ('text-align', 'center'),
            ('border', '1px solid #ccc'),
            ('padding', '6px')
        ]},
        {'selector': 'td', 'props': [
            ('text-align', 'center'),
            ('font-size', '12.5px'),
            ('color', '#222'),
            ('padding', '6px'),
            ('border', '1px solid #ddd')
        ]},
        {'selector': 'tr:nth-child(even)', 'props': [
            ('background-color', '#fafafa')
        ]},
        {'selector': 'tr:nth-child(odd)', 'props': [
            ('background-color', '#ffffff')
        ]},
        {'selector': 'tr:hover', 'props': [
            ('background-color', '#f5f5f5')
        ]}
    ])
    .format({
        'object_orientation_angle': '{:.2f}°',
        'von_mises_mean_orientation_deg': '{:.2f}°',
        'avg_orientation_angle_deg': '{:.2f}°'
    })
)

# ---- Display nicely below the Jupyter cell ----
display(styled_table)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from tabulate import tabulate

density_freq = "density_freq"
spacing_um = "spacing_um"
ff_col= 'fft_orientation_angle'
obj_orientation_col= "object_orientation_angle"
corr_col = obj_orientation_col
# Merge the dataframes on filename
merged_df = pd.merge(df[['filename', corr_col]], 
                    csv_df, 
                    on='filename', 
                    how='inner')

print(tabulate(merged_df.head(), headers='keys', tablefmt='psql'))



# Orientation-related columns that are most likely to correlate
orientation_columns = [
    'von_mises_mean_orientation_deg',
    'nematic_director_angle_deg', 
    'avg_orientation_angle_deg',
    'std_orientation_angle_deg',
    'nematic_order_parameter'
]

# Calculate correlations
correlation_results = []
for col in orientation_columns:
    if col in merged_df.columns:
        # Remove NaN values for correlation calculation
        valid_data = merged_df[[corr_col, col]].dropna()
        
        if len(valid_data) > 0:
            pearson_corr, pearson_p = pearsonr(valid_data[corr_col], valid_data[col])
            spearman_corr, spearman_p = spearmanr(valid_data[corr_col], valid_data[col])
            
            correlation_results.append({
                'column': col,
                'pearson_correlation': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_correlation': spearman_corr,
                'spearman_p_value': spearman_p,
                'n_samples': len(valid_data)
            })

# Create correlation results dataframe
corr_df = pd.DataFrame(correlation_results)
corr_df = corr_df.sort_values('pearson_correlation', key=abs, ascending=False)
print("Correlation with orientation-related columns:")
print(corr_df.round(4))

# Plot top correlations
top_correlations = corr_df.head(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, (_, row) in enumerate(top_correlations.iterrows()):
    ax = axes[idx]
    col = row['column']
    
    # Scatter plot
    ax.scatter(merged_df[corr_col], merged_df[col], alpha=0.6)
    ax.set_xlabel('FFT Orientation Angle')
    ax.set_ylabel(col)
    ax.set_title(f'Correlation: {row["pearson_correlation"]:.3f}')
    
    # Add trend line
    z = np.polyfit(merged_df[corr_col].dropna(), 
                   merged_df[col].dropna(), 1)
    p = np.poly1d(z)
    ax.plot(merged_df[corr_col], 
            p(merged_df[corr_col]), "r--", alpha=0.8)

plt.tight_layout()
plt.show()

# Density-related columns
density_columns = [
    'horizontal_density_cnts_per_um',
    'vertical_density_cnts_per_um', 
    'cnt_density_per_um2',
    'line_density_horizontal_lines_mean_intersections_per_row',
    'line_density_vertical_lines_mean_intersections_per_row'
]

density_correlations = []
for col in density_columns:
    if col in merged_df.columns:
        valid_data = merged_df[[corr_col, col]].dropna()
        
        if len(valid_data) > 0:
            pearson_corr, pearson_p = pearsonr(valid_data[corr_col], valid_data[col])
            spearman_corr, spearman_p = spearmanr(valid_data[corr_col], valid_data[col])
            
            density_correlations.append({
                'column': col,
                'pearson_correlation': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_correlation': spearman_corr,
                'spearman_p_value': spearman_p
            })

density_corr_df = pd.DataFrame(density_correlations)
density_corr_df = density_corr_df.sort_values('pearson_correlation', key=abs, ascending=False)
print("\nCorrelation with density-related columns:")
print(density_corr_df.round(4))

# Select key columns for comprehensive correlation analysis
key_columns = [
    corr_col,
    'von_mises_mean_orientation_deg',
    'nematic_director_angle_deg',
    'avg_orientation_angle_deg', 
    'nematic_order_parameter',
    'horizontal_density_cnts_per_um',
    'vertical_density_cnts_per_um',
    'cnt_density_per_um2'
]

# Filter to columns that exist in the merged dataframe
available_columns = [col for col in key_columns if col in merged_df.columns]
correlation_matrix = merged_df[available_columns].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0,
            square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix: FFT Orientation Angle vs Key Metrics')
plt.tight_layout()
plt.show()

# Statistical summary
print("Statistical Summary:")
print(f"FFT Orientation Angle - Mean: {merged_df[corr_col].mean():.2f}")
print(f"FFT Orientation Angle - Std: {merged_df[corr_col].std():.2f}")
print(f"FFT Orientation Angle - Range: {merged_df[corr_col].min():.2f} to {merged_df[corr_col].max():.2f}")

# Interpretation of results
significant_correlations = corr_df[corr_df['pearson_p_value'] < 0.05]

print(f"\nSignificant correlations (p < 0.05): {len(significant_correlations)}")
if len(significant_correlations) > 0:
    print("Most strongly correlated columns:")
    for _, row in significant_correlations.head().iterrows():
        strength = "strong" if abs(row['pearson_correlation']) > 0.5 else "moderate" if abs(row['pearson_correlation']) > 0.3 else "weak"
        direction = "positive" if row['pearson_correlation'] > 0 else "negative"
        print(f"  {row['column']}: {strength} {direction} correlation (r = {row['pearson_correlation']:.3f})")

# If orientation angles are circular (0-180 degrees), consider circular statistics
from scipy.stats import circmean, circstd

def circular_correlation(angles1, angles2):
    """Calculate circular correlation for angular data"""
    # Convert to radians
    angles1_rad = np.radians(angles1)
    angles2_rad = np.radians(angles2)
    
    # Calculate circular correlation coefficient
    n = len(angles1_rad)
    sin_diff1 = np.sin(angles1_rad - circmean(angles1_rad))
    sin_diff2 = np.sin(angles2_rad - circmean(angles2_rad))
    
    numerator = np.sum(sin_diff1 * sin_diff2)
    denominator = np.sqrt(np.sum(sin_diff1**2) * np.sum(sin_diff2**2))
    
    return numerator / denominator

# Apply circular correlation to orientation columns
print("\nCircular Correlation Analysis:")
for col in ['von_mises_mean_orientation_deg', 'nematic_director_angle_deg', 'avg_orientation_angle_deg']:
    if col in merged_df.columns:
        valid_data = merged_df[[corr_col, col]].dropna()
        if len(valid_data) > 10:  # Minimum sample size
            circ_corr = circular_correlation(valid_data[corr_col], valid_data[col])
            print(f"FFT vs {col}: Circular correlation = {circ_corr:.3f}")

# Correlate Orientations 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from tabulate import tabulate

density_freq = "density_freq"
spacing_um = "spacing_um"
ff_col= 'fft_orientation_angle'
obj_orientation_col= "object_orientation_angle"
corr_col = obj_orientation_col
# Merge the dataframes on filename
merged_df = pd.merge(df[['filename', corr_col]], 
                    csv_df, 
                    on='filename', 
                    how='inner')

print(tabulate(merged_df.head(), headers='keys', tablefmt='psql'))



# Orientation-related columns that are most likely to correlate
orientation_columns = [
    'von_mises_mean_orientation_deg',
    'nematic_director_angle_deg', 
    'avg_orientation_angle_deg',
    'std_orientation_angle_deg',
    'nematic_order_parameter'
]

# Calculate correlations
correlation_results = []
for col in orientation_columns:
    if col in merged_df.columns:
        # Remove NaN values for correlation calculation
        valid_data = merged_df[[corr_col, col]].dropna()
        
        if len(valid_data) > 0:
            pearson_corr, pearson_p = pearsonr(valid_data[corr_col], valid_data[col])
            spearman_corr, spearman_p = spearmanr(valid_data[corr_col], valid_data[col])
            
            correlation_results.append({
                'column': col,
                'pearson_correlation': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_correlation': spearman_corr,
                'spearman_p_value': spearman_p,
                'n_samples': len(valid_data)
            })

# Create correlation results dataframe
corr_df = pd.DataFrame(correlation_results)
corr_df = corr_df.sort_values('pearson_correlation', key=abs, ascending=False)
print("Correlation with orientation-related columns:")
print(corr_df.round(4))

# Plot top correlations
top_correlations = corr_df.head(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, (_, row) in enumerate(top_correlations.iterrows()):
    ax = axes[idx]
    col = row['column']
    
    # Scatter plot
    ax.scatter(merged_df[corr_col], merged_df[col], alpha=0.6)
    ax.set_xlabel('FFT Orientation Angle')
    ax.set_ylabel(col)
    ax.set_title(f'Correlation: {row["pearson_correlation"]:.3f}')
    
    # Add trend line
    z = np.polyfit(merged_df[corr_col].dropna(), 
                   merged_df[col].dropna(), 1)
    p = np.poly1d(z)
    ax.plot(merged_df[corr_col], 
            p(merged_df[corr_col]), "r--", alpha=0.8)

plt.tight_layout()
plt.show()

# Density-related columns
density_columns = [
    'horizontal_density_cnts_per_um',
    'vertical_density_cnts_per_um', 
    'cnt_density_per_um2',
    'line_density_horizontal_lines_mean_intersections_per_row',
    'line_density_vertical_lines_mean_intersections_per_row'
]

density_correlations = []
for col in density_columns:
    if col in merged_df.columns:
        valid_data = merged_df[[corr_col, col]].dropna()
        
        if len(valid_data) > 0:
            pearson_corr, pearson_p = pearsonr(valid_data[corr_col], valid_data[col])
            spearman_corr, spearman_p = spearmanr(valid_data[corr_col], valid_data[col])
            
            density_correlations.append({
                'column': col,
                'pearson_correlation': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_correlation': spearman_corr,
                'spearman_p_value': spearman_p
            })

density_corr_df = pd.DataFrame(density_correlations)
density_corr_df = density_corr_df.sort_values('pearson_correlation', key=abs, ascending=False)
print("\nCorrelation with density-related columns:")
print(density_corr_df.round(4))

# Select key columns for comprehensive correlation analysis
key_columns = [
    corr_col,
    'von_mises_mean_orientation_deg',
    'nematic_director_angle_deg',
    'avg_orientation_angle_deg', 
    'nematic_order_parameter',
    'horizontal_density_cnts_per_um',
    'vertical_density_cnts_per_um',
    'cnt_density_per_um2'
]

# Filter to columns that exist in the merged dataframe
available_columns = [col for col in key_columns if col in merged_df.columns]
correlation_matrix = merged_df[available_columns].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='RdBu_r', center=0,
            square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix: FFT Orientation Angle vs Key Metrics')
plt.tight_layout()
plt.show()

# Statistical summary
print("Statistical Summary:")
print(f"FFT Orientation Angle - Mean: {merged_df[corr_col].mean():.2f}")
print(f"FFT Orientation Angle - Std: {merged_df[corr_col].std():.2f}")
print(f"FFT Orientation Angle - Range: {merged_df[corr_col].min():.2f} to {merged_df[corr_col].max():.2f}")

# Interpretation of results
significant_correlations = corr_df[corr_df['pearson_p_value'] < 0.05]

print(f"\nSignificant correlations (p < 0.05): {len(significant_correlations)}")
if len(significant_correlations) > 0:
    print("Most strongly correlated columns:")
    for _, row in significant_correlations.head().iterrows():
        strength = "strong" if abs(row['pearson_correlation']) > 0.5 else "moderate" if abs(row['pearson_correlation']) > 0.3 else "weak"
        direction = "positive" if row['pearson_correlation'] > 0 else "negative"
        print(f"  {row['column']}: {strength} {direction} correlation (r = {row['pearson_correlation']:.3f})")

# If orientation angles are circular (0-180 degrees), consider circular statistics
from scipy.stats import circmean, circstd

def circular_correlation(angles1, angles2):
    """Calculate circular correlation for angular data"""
    # Convert to radians
    angles1_rad = np.radians(angles1)
    angles2_rad = np.radians(angles2)
    
    # Calculate circular correlation coefficient
    n = len(angles1_rad)
    sin_diff1 = np.sin(angles1_rad - circmean(angles1_rad))
    sin_diff2 = np.sin(angles2_rad - circmean(angles2_rad))
    
    numerator = np.sum(sin_diff1 * sin_diff2)
    denominator = np.sqrt(np.sum(sin_diff1**2) * np.sum(sin_diff2**2))
    
    return numerator / denominator

# Apply circular correlation to orientation columns
print("\nCircular Correlation Analysis:")
for col in ['von_mises_mean_orientation_deg', 'nematic_director_angle_deg', 'avg_orientation_angle_deg']:
    if col in merged_df.columns:
        valid_data = merged_df[[corr_col, col]].dropna()
        if len(valid_data) > 10:  # Minimum sample size
            circ_corr = circular_correlation(valid_data[corr_col], valid_data[col])
            print(f"FFT vs {col}: Circular correlation = {circ_corr:.3f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import circmean, circstd, circvar
import seaborn as sns

def circular_difference(angle1, angle2, period=180):
    """
    Calculate the minimal circular difference between two angles (0-180° range)
    Returns difference in degrees (-90° to +90°)
    """
    diff = angle1 - angle2
    # Wrap differences to the range [-period/2, period/2]
    diff = np.mod(diff + period/2, period) - period/2
    return diff

def circular_distance(angle1, angle2, period=180):
    """
    Calculate absolute circular distance (always positive, 0-90°)
    """
    diff = np.abs(angle1 - angle2)
    return np.minimum(diff, period - diff)

def circular_descriptive_stats(angles, period=180):
    """
    Calculate circular descriptive statistics
    """
    angles_rad = np.radians(angles) * 2  # Convert to radians and double for 0-180° range
    mean_circ = np.degrees(circmean(angles_rad)) / 2
    std_circ = np.degrees(circstd(angles_rad)) / 2
    var_circ = np.degrees(circvar(angles_rad)) / 2
    
    return {
        'circular_mean': mean_circ,
        'circular_std': std_circ,
        'circular_variance': var_circ,
        'linear_mean': np.mean(angles),
        'linear_std': np.std(angles)
    }

# Calculate circular differences
valid_data = merged_df[[corr_col, 'von_mises_mean_orientation_deg', 'avg_orientation_angle_deg']].dropna()

print("="*60)
print("CIRCULAR ORIENTATION DIFFERENCE ANALYSIS")
print("="*60)

# 1. Calculate differences between methods
valid_data['fft_vs_von_mises_diff'] = circular_difference(valid_data[corr_col], valid_data['von_mises_mean_orientation_deg'])
valid_data['fft_vs_avg_diff'] = circular_difference(valid_data[corr_col], valid_data['avg_orientation_angle_deg'])
valid_data['von_mises_vs_avg_diff'] = circular_difference(valid_data['von_mises_mean_orientation_deg'], valid_data['avg_orientation_angle_deg'])

# 2. Calculate absolute distances
valid_data['fft_vs_von_mises_dist'] = circular_distance(valid_data[corr_col], valid_data['von_mises_mean_orientation_deg'])
valid_data['fft_vs_avg_dist'] = circular_distance(valid_data[corr_col], valid_data['avg_orientation_angle_deg'])
valid_data['von_mises_vs_avg_dist'] = circular_distance(valid_data['von_mises_mean_orientation_deg'], valid_data['avg_orientation_angle_deg'])

# 3. Print comprehensive difference statistics
print(f"\nSample size: {len(valid_data)}")
print("\nABSOLUTE CIRCULAR DISTANCES (0-90° range):")
print(f"FFT vs Von Mises: {valid_data['fft_vs_von_mises_dist'].mean():.2f}° ± {valid_data['fft_vs_von_mises_dist'].std():.2f}°")
print(f"FFT vs Avg Orientation: {valid_data['fft_vs_avg_dist'].mean():.2f}° ± {valid_data['fft_vs_avg_dist'].std():.2f}°")
print(f"Von Mises vs Avg Orientation: {valid_data['von_mises_vs_avg_dist'].mean():.2f}° ± {valid_data['von_mises_vs_avg_dist'].std():.2f}°")

print("\nSIGNED CIRCULAR DIFFERENCES (-90° to +90° range):")
print(f"FFT vs Von Mises: {valid_data['fft_vs_von_mises_diff'].mean():.2f}° ± {valid_data['fft_vs_von_mises_diff'].std():.2f}°")
print(f"FFT vs Avg Orientation: {valid_data['fft_vs_avg_diff'].mean():.2f}° ± {valid_data['fft_vs_avg_diff'].std():.2f}°")

# 4. Create visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Distribution of absolute distances
distance_columns = ['fft_vs_von_mises_dist', 'fft_vs_avg_dist', 'von_mises_vs_avg_dist']
distance_labels = ['FFT vs Von Mises', 'FFT vs Avg', 'Von Mises vs Avg']

for i, (col, label) in enumerate(zip(distance_columns, distance_labels)):
    axes[0,0].hist(valid_data[col], alpha=0.7, label=label, bins=20)
axes[0,0].set_xlabel('Absolute Circular Distance (°)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('Distribution of Absolute Differences')
axes[0,0].legend()
axes[0,0].axvline(valid_data[col].mean(), color='red', linestyle='--', alpha=0.7, label='Mean')

# Plot 2: Signed differences
sns.violinplot(data=valid_data[['fft_vs_von_mises_diff', 'fft_vs_avg_diff']], 
               ax=axes[0,1])
axes[0,1].set_ylabel('Signed Circular Difference (°)')
axes[0,1].set_title('Signed Differences (FFT - Other)')
axes[0,1].axhline(0, color='red', linestyle='--', alpha=0.7)

# Plot 3: Scatter plot with circular wrapping
axes[0,2].scatter(valid_data[corr_col], valid_data['von_mises_mean_orientation_deg'], 
                 alpha=0.6, label='FFT vs Von Mises', s=30)
axes[0,2].scatter(valid_data[corr_col], valid_data['avg_orientation_angle_deg'], 
                 alpha=0.6, label='FFT vs Avg', s=30)
axes[0,2].plot([0, 180], [0, 180], 'k--', alpha=0.5, label='Perfect agreement')
axes[0,2].set_xlabel('FFT Orientation Angle (°)')
axes[0,2].set_ylabel('Other Orientation Angle (°)')
axes[0,2].set_title('Orientation Angle Comparison')
axes[0,2].legend()
axes[0,2].set_xlim(0, 180)
axes[0,2].set_ylim(0, 180)

# Plot 4: Circular mean and spread for each method
methods_data = {
    'FFT': valid_data[corr_col],
    'Von Mises': valid_data['von_mises_mean_orientation_deg'],
    'Avg Orientation': valid_data['avg_orientation_angle_deg']
}

methods_stats = {}
for method, angles in methods_data.items():
    stats = circular_descriptive_stats(angles)
    methods_stats[method] = stats

# Create circular histogram
for i, (method, angles) in enumerate(methods_data.items()):
    axes[1,0].hist(angles, alpha=0.6, label=method, bins=36, range=(0, 180))
axes[1,0].set_xlabel('Orientation Angle (°)')
axes[1,0].set_ylabel('Frequency')
axes[1,0].set_title('Distribution of Orientation Methods')
axes[1,0].legend()

# Plot 5: Circular standard deviation comparison
std_values = [stats['circular_std'] for stats in methods_stats.values()]
methods = list(methods_stats.keys())
axes[1,1].bar(methods, std_values, alpha=0.7, color=['skyblue', 'lightcoral', 'lightgreen'])
axes[1,1].set_ylabel('Circular Standard Deviation (°)')
axes[1,1].set_title('Circular Spread of Each Method')
for i, v in enumerate(std_values):
    axes[1,1].text(i, v + 0.1, f'{v:.2f}°', ha='center')

# Plot 6: Agreement analysis
agreement_threshold = 10  # degrees
agreement_data = []
for col in distance_columns[:2]:  # Only FFT comparisons
    within_threshold = (valid_data[col] <= agreement_threshold).mean() * 100
    agreement_data.append(within_threshold)

axes[1,2].bar(['FFT vs Von Mises', 'FFT vs Avg'], agreement_data, 
              color=['lightblue', 'lightgreen'], alpha=0.7)
axes[1,2].set_ylabel(f'Percentage within {agreement_threshold}° (%)')
axes[1,2].set_title('Agreement Between Methods')
axes[1,2].set_ylim(0, 100)
for i, v in enumerate(agreement_data):
    axes[1,2].text(i, v + 2, f'{v:.1f}%', ha='center')

plt.tight_layout()
plt.show()

# 5. Print detailed statistics
print("\n" + "="*60)
print("DETAILED CIRCULAR STATISTICS FOR EACH METHOD")
print("="*60)

for method, stats in methods_stats.items():
    print(f"\n{method}:")
    print(f"  Circular Mean: {stats['circular_mean']:.2f}°")
    print(f"  Circular Std: {stats['circular_std']:.2f}°")
    print(f"  Linear Mean: {stats['linear_mean']:.2f}°")
    print(f"  Linear Std: {stats['linear_std']:.2f}°")

# 6. Agreement analysis
print("\n" + "="*60)
print("AGREEMENT ANALYSIS")
print("="*60)

thresholds = [5, 10, 15, 20]  # degrees
for threshold in thresholds:
    print(f"\nWithin {threshold}° agreement:")
    fft_von_mises_agree = (valid_data['fft_vs_von_mises_dist'] <= threshold).mean() * 100
    fft_avg_agree = (valid_data['fft_vs_avg_dist'] <= threshold).mean() * 100
    print(f"  FFT vs Von Mises: {fft_von_mises_agree:.1f}%")
    print(f"  FFT vs Avg: {fft_avg_agree:.1f}%")

# 7. Systematic bias analysis
print("\n" + "="*60)
print("SYSTEMATIC BIAS ANALYSIS")
print("="*60)

fft_von_mises_bias = valid_data['fft_vs_von_mises_diff'].mean()
fft_avg_bias = valid_data['fft_vs_avg_diff'].mean()

print(f"FFT tends to be {abs(fft_von_mises_bias):.2f}° {'higher' if fft_von_mises_bias > 0 else 'lower'} than Von Mises")
print(f"FFT tends to be {abs(fft_avg_bias):.2f}° {'higher' if fft_avg_bias > 0 else 'lower'} than Average Orientation")

# 8. Outlier analysis
large_disagreement = valid_data[valid_data['fft_vs_von_mises_dist'] > 30]  # > 30° difference
if len(large_disagreement) > 0:
    print(f"\nSamples with large disagreement (>30°) between FFT and Von Mises: {len(large_disagreement)}")
    print("These might represent cases where the methods fundamentally disagree")

# Correlate Density frequency

In [ ]:
merged_df = pd.merge(df[['filename', 'density_freq', 'spacing_um']], 
                    csv_df, 
                    on='filename', 
                    how='inner')

print(f"Merged dataframe shape: {merged_df.shape}")
print(f"Available columns: {merged_df.columns.tolist()}")

# Target columns from csv_df
target_columns = [
    'line_density_horizontal_lines_mean_intersections_per_row',
    'horizontal_density_cnts_per_um',
    'line_density_vertical_lines_mean_intersections_per_row',
    'vertical_density_cnts_per_um',
    'num_cnts',
    'cnt_density_per_um2',
    'avg_length_pixels',
    'avg_width_pixels',
    'avg_area_pixels2',
    'avg_perimeter_pixels'
]

# Filter to columns that actually exist in the merged dataframe
available_targets = [col for col in target_columns if col in merged_df.columns]
print(f"Analyzing {len(available_targets)} target columns:")
print(available_targets)


# Analyze correlations with density_freq
density_freq_correlations = []

for col in available_targets:
    # Remove NaN values for correlation calculation
    valid_data = merged_df[['density_freq', col]].dropna()
    
    if len(valid_data) > 3:  # Minimum sample size requirement
        try:
            pearson_corr, pearson_p = pearsonr(valid_data['density_freq'], valid_data[col])
            spearman_corr, spearman_p = spearmanr(valid_data['density_freq'], valid_data[col])
            
            density_freq_correlations.append({
                'target_column': col,
                'pearson_correlation': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_correlation': spearman_corr,
                'spearman_p_value': spearman_p,
                'n_samples': len(valid_data)
            })
        except:
            print(f"Could not calculate correlation for {col}")

# Create and sort results dataframe
density_freq_corr_df = pd.DataFrame(density_freq_correlations)
density_freq_corr_df = density_freq_corr_df.sort_values('pearson_correlation', key=abs, ascending=False)

print("Correlation with density_freq:")
print(density_freq_corr_df.round(4))

# Analyze correlations with spacing_um
spacing_correlations = []

for col in available_targets:
    # Remove NaN values for correlation calculation
    valid_data = merged_df[['spacing_um', col]].dropna()
    
    if len(valid_data) > 3:  # Minimum sample size requirement
        try:
            pearson_corr, pearson_p = pearsonr(valid_data['spacing_um'], valid_data[col])
            spearman_corr, spearman_p = spearmanr(valid_data['spacing_um'], valid_data[col])
            
            spacing_correlations.append({
                'target_column': col,
                'pearson_correlation': pearson_corr,
                'pearson_p_value': pearson_p,
                'spearman_correlation': spearman_corr,
                'spearman_p_value': spearman_p,
                'n_samples': len(valid_data)
            })
        except:
            print(f"Could not calculate correlation for {col}")

# Create and sort results dataframe
spacing_corr_df = pd.DataFrame(spacing_correlations)
spacing_corr_df = spacing_corr_df.sort_values('pearson_correlation', key=abs, ascending=False)

print("\nCorrelation with spacing_um:")
print(spacing_corr_df.round(4))

# Plot top 4 correlations for density_freq
top_density_corr = density_freq_corr_df.head(4)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (_, row) in enumerate(top_density_corr.iterrows()):
    col = row['target_column']
    
    # Scatter plot
    axes[idx].scatter(merged_df['density_freq'], merged_df[col], alpha=0.6, s=50)
    axes[idx].set_xlabel('Density Frequency')
    axes[idx].set_ylabel(col)
    axes[idx].set_title(f'{col}\nPearson r: {row["pearson_correlation"]:.3f}')
    
    # Add trend line if correlation is meaningful
    if abs(row['pearson_correlation']) > 0.1:
        z = np.polyfit(merged_df['density_freq'].dropna(), 
                       merged_df[col].dropna(), 1)
        p = np.poly1d(z)
        x_range = np.linspace(merged_df['density_freq'].min(), merged_df['density_freq'].max(), 100)
        axes[idx].plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)

plt.tight_layout()
plt.suptitle('Top Correlations with Density Frequency', y=1.02, fontsize=14)
plt.show()

# Plot top 4 correlations for spacing_um
top_spacing_corr = spacing_corr_df.head(4)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (_, row) in enumerate(top_spacing_corr.iterrows()):
    col = row['target_column']
    
    # Scatter plot
    axes[idx].scatter(merged_df['spacing_um'], merged_df[col], alpha=0.6, s=50)
    axes[idx].set_xlabel('Spacing (μm)')
    axes[idx].set_ylabel(col)
    axes[idx].set_title(f'{col}\nPearson r: {row["pearson_correlation"]:.3f}')
    
    # Add trend line if correlation is meaningful
    if abs(row['pearson_correlation']) > 0.1:
        z = np.polyfit(merged_df['spacing_um'].dropna(), 
                       merged_df[col].dropna(), 1)
        p = np.poly1d(z)
        x_range = np.linspace(merged_df['spacing_um'].min(), merged_df['spacing_um'].max(), 100)
        axes[idx].plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)

plt.tight_layout()
plt.suptitle('Top Correlations with Spacing (μm)', y=1.02, fontsize=14)
plt.show()


# Create correlation matrix for key variables
key_vars = ['density_freq', 'spacing_um'] + available_targets
correlation_matrix = merged_df[key_vars].corr()

# Plot heatmap
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))  # Mask upper triangle
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            square=True, fmt='.3f', cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
plt.title('Correlation Matrix: Density Frequency & Spacing vs Target Metrics', fontsize=14)
plt.tight_layout()
plt.show()

# Statistical summary
print("Statistical Summary:")
print(f"Density Frequency - Mean: {merged_df['density_freq'].mean():.3f} ± {merged_df['density_freq'].std():.3f}")
print(f"Spacing (μm) - Mean: {merged_df['spacing_um'].mean():.3f} ± {merged_df['spacing_um'].std():.3f}")

# Interpretation for density_freq
sig_density = density_freq_corr_df[density_freq_corr_df['pearson_p_value'] < 0.05]
print(f"\nSignificant correlations for density_freq (p < 0.05): {len(sig_density)}")
if len(sig_density) > 0:
    print("Most significant correlations:")
    for _, row in sig_density.head(3).iterrows():
        strength = "strong" if abs(row['pearson_correlation']) > 0.7 else "moderate" if abs(row['pearson_correlation']) > 0.3 else "weak"
        direction = "positive" if row['pearson_correlation'] > 0 else "negative"
        print(f"  {row['target_column']}: {strength} {direction} correlation (r = {row['pearson_correlation']:.3f}, p = {row['pearson_p_value']:.4f})")

# Interpretation for spacing_um
sig_spacing = spacing_corr_df[spacing_corr_df['pearson_p_value'] < 0.05]
print(f"\nSignificant correlations for spacing_um (p < 0.05): {len(sig_spacing)}")
if len(sig_spacing) > 0:
    print("Most significant correlations:")
    for _, row in sig_spacing.head(3).iterrows():
        strength = "strong" if abs(row['pearson_correlation']) > 0.7 else "moderate" if abs(row['pearson_correlation']) > 0.3 else "weak"
        direction = "positive" if row['pearson_correlation'] > 0 else "negative"
        print(f"  {row['target_column']}: {strength} {direction} correlation (r = {row['pearson_correlation']:.3f}, p = {row['pearson_p_value']:.4f})")


# Check if density_freq and spacing_um are related (they should be inversely related)
if 'density_freq' in merged_df.columns and 'spacing_um' in merged_df.columns:
    valid_data = merged_df[['density_freq', 'spacing_um']].dropna()
    if len(valid_data) > 0:
        corr, p_value = pearsonr(valid_data['density_freq'], valid_data['spacing_um'])
        print(f"\nRelationship between density_freq and spacing_um:")
        print(f"Pearson correlation: {corr:.3f} (p = {p_value:.4f})")
        
        # Expected: density_freq should be inversely proportional to spacing
        expected_inverse = -1.0 * (merged_df['density_freq'] * merged_df['spacing_um']).mean()
        print(f"Expected inverse relationship constant: ~{expected_inverse:.3f}")




# Compare GT and Input image FFT based features

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
from scipy.signal import find_peaks
from scipy.stats import pearsonr, spearmanr
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

def fft_analysis(image, title_prefix=""):
    """Perform FFT analysis and return features as dictionary."""
    if image.ndim == 3 and image.shape[2] == 3:
        image = image.mean(axis=2)
    if image.ndim > 2:
        image = image[0]
    image = image.astype(np.float32)

    # For masks, binarize
    if title_prefix.lower().startswith("mask"):
        image = (image > 0).astype(np.uint8)

    # FFT computation
    f = np.fft.fft2(image)
    fshift = np.fft.fftshift(f)
    fshift[image.shape[0]//2, image.shape[1]//2] = 0
    magnitude = np.abs(fshift)

    h, w = image.shape
    cy, cx = h//2, w//2

    # Polar coordinates
    Y, X = np.indices((h, w))
    R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)
    theta = np.arctan2(Y-cy, X-cx)
    theta_deg = (np.rad2deg(theta) % 180)

    # Radial profile
    radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
    r = np.arange(len(radial_profile))

    peaks, _ = find_peaks(radial_profile, distance=5)
    main_r = peaks[np.argmax(radial_profile[peaks])] if len(peaks) > 0 else None
    density_freq = main_r / w if main_r is not None else None

    # Angular profile
    angular_bins = 180
    hist, edges = np.histogram(theta_deg, bins=angular_bins, weights=magnitude)
    orientation_angle = edges[np.argmax(hist)]
    orientation_angle = (orientation_angle + 90) % 180

    # Alignment Index
    alignment_index = np.max(hist) / np.mean(hist)

    # Order Parameter
    if main_r:
        P_peak = radial_profile[main_r]
        mask_bg = (r > main_r + 10) & (r < len(r)//2)
        P_bg = np.mean(radial_profile[mask_bg]) if np.any(mask_bg) else 1
        order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
    else:
        order_parameter = None

    # Physical spacing
    field_um_x, field_um_y = 5.0, 5.0
    px_size_um_x = field_um_x / w
    px_size_um_y = field_um_y / h
    px_size_um = 0.5 * (px_size_um_x + px_size_um_y)

    if density_freq and density_freq > 0:
        spacing_px = 1.0 / density_freq
        spacing_um = spacing_px * px_size_um
    else:
        spacing_px = spacing_um = None

    # Return features as dictionary
    features = {
        'density_freq': density_freq,
        'orientation_angle': orientation_angle,
        'alignment_index': alignment_index,
        'order_parameter': order_parameter,
        'spacing_px': spacing_px,
        'spacing_um': spacing_um,
        'radial_profile': radial_profile,
        'angular_profile': hist
    }
    
    return features, magnitude, image

def process_all_images(original_folder, mask_folder):
    """Process all images in both folders and extract features."""
    original_features = []
    mask_features = []
    filenames = []
    
    # Get common files
    original_files = set(os.listdir(original_folder))
    mask_files = set(os.listdir(mask_folder))
    common_files = original_files.intersection(mask_files)
    
    print(f"Found {len(common_files)} common files to process")
    
    for filename in common_files:
        if filename.endswith(('.tif', '.tiff', '.png', '.jpg')):
            try:
                # Load images
                original_img = tiff.imread(os.path.join(original_folder, filename))
                mask_img = tiff.imread(os.path.join(mask_folder, filename))
                
                # Extract features
                orig_feat, _, _ = fft_analysis(original_img, f"Original_{filename}")
                mask_feat, _, _ = fft_analysis(mask_img, f"Mask_{filename}")
                
                original_features.append(orig_feat)
                mask_features.append(mask_feat)
                filenames.append(filename)
                
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    
    return original_features, mask_features, filenames

def create_comparison_dataframe(original_features, mask_features, filenames):
    """Create a DataFrame for easy comparison of features."""
    comparison_data = []
    
    for i, (orig, mask, fname) in enumerate(zip(original_features, mask_features, filenames)):
        row = {
            'filename': fname,
            'orig_density_freq': orig['density_freq'],
            'mask_density_freq': mask['density_freq'],
            'orig_orientation': orig['orientation_angle'],
            'mask_orientation': mask['orientation_angle'],
            'orig_alignment': orig['alignment_index'],
            'mask_alignment': mask['alignment_index'],
            'orig_order_param': orig['order_parameter'],
            'mask_order_param': mask['order_parameter'],
            'orig_spacing_um': orig['spacing_um'],
            'mask_spacing_um': mask['spacing_um']
        }
        comparison_data.append(row)
    
    return pd.DataFrame(comparison_data)

def statistical_comparison(df):
    """Perform statistical comparison between original and mask features."""
    features = ['density_freq', 'orientation', 'alignment', 'order_param', 'spacing_um']
    
    print("="*60)
    print("STATISTICAL COMPARISON")
    print("="*60)
    
    for feature in features:
        orig_col = f'orig_{feature}'
        mask_col = f'mask_{feature}'
        
        # Remove None values for comparison
        valid_idx = df[[orig_col, mask_col]].notna().all(axis=1)
        orig_vals = df.loc[valid_idx, orig_col]
        mask_vals = df.loc[valid_idx, mask_col]
        
        if len(orig_vals) > 0:
            mae = mean_absolute_error(orig_vals, mask_vals)
            rmse = np.sqrt(mean_squared_error(orig_vals, mask_vals))
            r2 = r2_score(orig_vals, mask_vals)
            pearson_corr, p_val_pearson = pearsonr(orig_vals, mask_vals)
            spearman_corr, p_val_spearman = spearmanr(orig_vals, mask_vals)
            
            print(f"\n{feature.upper()} Comparison:")
            print(f"  MAE: {mae:.4f}")
            print(f"  RMSE: {rmse:.4f}")
            print(f"  R²: {r2:.4f}")
            print(f"  Pearson r: {pearson_corr:.4f} (p={p_val_pearson:.4f})")
            print(f"  Spearman ρ: {spearman_corr:.4f} (p={p_val_spearman:.4f})")

def create_bland_altman_plots(df):
    """Create Bland-Altman plots for each feature."""
    features = ['density_freq', 'orientation', 'alignment', 'order_param', 'spacing_um']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, feature in enumerate(features):
        if i >= len(axes):
            break
            
        orig_col = f'orig_{feature}'
        mask_col = f'mask_{feature}'
        
        # Remove None values
        valid_idx = df[[orig_col, mask_col]].notna().all(axis=1)
        orig_vals = df.loc[valid_idx, orig_col]
        mask_vals = df.loc[valid_idx, mask_col]
        
        if len(orig_vals) > 0:
            means = (orig_vals + mask_vals) / 2
            diffs = orig_vals - mask_vals
            
            axes[i].scatter(means, diffs, alpha=0.6)
            axes[i].axhline(y=np.mean(diffs), color='red', linestyle='--', label=f'Mean diff: {np.mean(diffs):.3f}')
            axes[i].axhline(y=np.mean(diffs) + 1.96*np.std(diffs), color='gray', linestyle='--', 
                          label='+1.96 SD')
            axes[i].axhline(y=np.mean(diffs) - 1.96*np.std(diffs), color='gray', linestyle='--', 
                          label='-1.96 SD')
            
            axes[i].set_xlabel(f'Mean of {feature}')
            axes[i].set_ylabel('Difference (Original - Mask)')
            axes[i].set_title(f'Bland-Altman: {feature}')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for i in range(len(features), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

def create_scatter_comparison(df):
    """Create scatter plots comparing original vs mask features."""
    features = ['density_freq', 'orientation', 'alignment', 'order_param', 'spacing_um']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, feature in enumerate(features):
        if i >= len(axes):
            break
            
        orig_col = f'orig_{feature}'
        mask_col = f'mask_{feature}'
        
        # Remove None values
        valid_idx = df[[orig_col, mask_col]].notna().all(axis=1)
        orig_vals = df.loc[valid_idx, orig_col]
        mask_vals = df.loc[valid_idx, mask_col]
        
        if len(orig_vals) > 0:
            axes[i].scatter(orig_vals, mask_vals, alpha=0.6)
            
            # Add perfect agreement line
            min_val = min(orig_vals.min(), mask_vals.min())
            max_val = max(orig_vals.max(), mask_vals.max())
            axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect agreement')
            
            # Add regression line
            z = np.polyfit(orig_vals, mask_vals, 1)
            p = np.poly1d(z)
            axes[i].plot(orig_vals, p(orig_vals), "b--", alpha=0.5, label='Regression line')
            
            r2 = r2_score(orig_vals, mask_vals)
            axes[i].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[i].transAxes, 
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
            
            axes[i].set_xlabel(f'Original {feature}')
            axes[i].set_ylabel(f'Mask {feature}')
            axes[i].set_title(f'{feature.title()} Comparison')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for i in range(len(features), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

def analyze_outliers(df, original_features, mask_features, filenames, threshold_std=2):
    """Identify and analyze outliers where features differ significantly."""
    features = ['density_freq', 'orientation', 'alignment', 'order_param', 'spacing_um']
    
    print("\n" + "="*60)
    print("OUTLIER ANALYSIS")
    print("="*60)
    
    outlier_info = []
    
    for feature in features:
        orig_col = f'orig_{feature}'
        mask_col = f'mask_{feature}'
        
        valid_idx = df[[orig_col, mask_col]].notna().all(axis=1)
        diffs = abs(df.loc[valid_idx, orig_col] - df.loc[valid_idx, mask_col])
        
        # Identify outliers (difference > threshold_std standard deviations from mean)
        outlier_threshold = np.mean(diffs) + threshold_std * np.std(diffs)
        outliers = diffs > outlier_threshold
        
        if outliers.any():
            outlier_indices = valid_idx[valid_idx][outliers].index
            print(f"\nOutliers in {feature}:")
            for idx in outlier_indices:
                diff_val = diffs.loc[idx]
                print(f"  {filenames[idx]}: difference = {diff_val:.4f}")
                
                outlier_info.append({
                    'filename': filenames[idx],
                    'feature': feature,
                    'difference': diff_val,
                    'original_value': df.loc[idx, orig_col],
                    'mask_value': df.loc[idx, mask_col]
                })
    
    return outlier_info

def visualize_outlier_pairs(outlier_info, original_folder, mask_folder, max_display=5):
    """Visualize original and mask pairs for outliers."""
    if not outlier_info:
        print("No outliers to display.")
        return
    
    # Get unique filenames from outliers
    unique_filenames = list(set([info['filename'] for info in outlier_info]))[:max_display]
    
    for filename in unique_filenames:
        try:
            # Load and process images
            original_img = tiff.imread(os.path.join(original_folder, filename))
            mask_img = tiff.imread(os.path.join(mask_folder, filename))
            
            # Get features for this file
            orig_feat, orig_magnitude, orig_processed = fft_analysis(original_img, f"Original_{filename}")
            mask_feat, mask_magnitude, mask_processed = fft_analysis(mask_img, f"Mask_{filename}")
            
            # Create detailed comparison plot
            fig, axes = plt.subplots(2, 4, figsize=(20, 10))
            fig.suptitle(f'Outlier Analysis: {filename}', fontsize=16, fontweight='bold')
            
            # Original image and FFT
            axes[0,0].imshow(orig_processed, cmap='gray')
            axes[0,0].set_title('Original Image')
            axes[0,0].axis('off')
            
            axes[0,1].imshow(np.log1p(orig_magnitude), cmap='gray')
            axes[0,1].set_title('Original FFT')
            axes[0,1].axis('off')
            
            # Mask image and FFT
            axes[1,0].imshow(mask_processed, cmap='gray')
            axes[1,0].set_title('Mask Image')
            axes[1,0].axis('off')
            
            axes[1,1].imshow(np.log1p(mask_magnitude), cmap='gray')
            axes[1,1].set_title('Mask FFT')
            axes[1,1].axis('off')
            
            # Radial profile comparison
            axes[0,2].plot(orig_feat['radial_profile'], 'b-', label='Original', alpha=0.7)
            axes[0,2].plot(mask_feat['radial_profile'], 'r-', label='Mask', alpha=0.7)
            axes[0,2].set_title('Radial Profile Comparison')
            axes[0,2].legend()
            axes[0,2].grid(True, alpha=0.3)
            
            # Angular profile comparison
            axes[1,2].plot(orig_feat['angular_profile'], 'b-', label='Original', alpha=0.7)
            axes[1,2].plot(mask_feat['angular_profile'], 'r-', label='Mask', alpha=0.7)
            axes[1,2].set_title('Angular Profile Comparison')
            axes[1,2].legend()
            axes[1,2].grid(True, alpha=0.3)
            
            # Feature differences
            feature_names = ['Density Freq', 'Orientation', 'Alignment', 'Order Param', 'Spacing (μm)']
            orig_values = [
                orig_feat['density_freq'] or 0,
                orig_feat['orientation_angle'],
                orig_feat['alignment_index'],
                orig_feat['order_parameter'] or 0,
                orig_feat['spacing_um'] or 0
            ]
            mask_values = [
                mask_feat['density_freq'] or 0,
                mask_feat['orientation_angle'],
                mask_feat['alignment_index'],
                mask_feat['order_parameter'] or 0,
                mask_feat['spacing_um'] or 0
            ]
            
            x_pos = np.arange(len(feature_names))
            width = 0.35
            
            axes[0,3].bar(x_pos - width/2, orig_values, width, label='Original', alpha=0.7)
            axes[0,3].bar(x_pos + width/2, mask_values, width, label='Mask', alpha=0.7)
            axes[0,3].set_title('Feature Values Comparison')
            axes[0,3].set_xticks(x_pos)
            axes[0,3].set_xticklabels(feature_names, rotation=45)
            axes[0,3].legend()
            
            # Differences
            differences = [abs(o - m) for o, m in zip(orig_values, mask_values)]
            axes[1,3].bar(feature_names, differences, alpha=0.7, color='orange')
            axes[1,3].set_title('Absolute Differences')
            axes[1,3].set_xticklabels(feature_names, rotation=45)
            
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Error visualizing outlier {filename}: {e}")

def main_comparison_analysis(original_folder, mask_folder):
    """Main function to run complete comparison analysis."""
    
    print("Starting comprehensive FFT feature comparison...")
    
    # Process all images
    original_features, mask_features, filenames = process_all_images(original_folder, mask_folder)
    
    if not original_features:
        print("No valid images found for comparison.")
        return
    
    # Create comparison dataframe
    df = create_comparison_dataframe(original_features, mask_features, filenames)
    
    # 1. Statistical comparison
    statistical_comparison(df)
    
    # 2. Bland-Altman plots
    create_bland_altman_plots(df)
    
    # 3. Scatter comparison plots
    create_scatter_comparison(df)
    
    # 4. Outlier analysis
    outlier_info = analyze_outliers(df, original_features, mask_features, filenames)
    
    # 5. Visualize outliers
    visualize_outlier_pairs(outlier_info, original_folder, mask_folder)
    
    # 6. Summary statistics
    print("\n" + "="*60)
    print("SUMMARY STATISTICS")
    print("="*60)
    print(f"Total images processed: {len(original_features)}")
    print(f"Features with good agreement (R² > 0.8): TBD")
    print(f"Features needing improvement: TBD")
    
    return df, original_features, mask_features, filenames

# Run the analysis
if __name__ == "__main__":
    mask_path = "../../data/annotations_uniques/test/masks"
    image_path = "../../data/annotations_uniques/test/images"
    
    # Run single image comparison (your original code)
    mask_filename = "400-1293-17-c10k1r5_pd_sp0_9_fl0_1_right.tif"
    mask_image_path = os.path.join(mask_path, mask_filename)
    original_image_path = os.path.join(image_path, mask_filename)
    
    # Load both
    mask = tiff.imread(mask_image_path)
    image = tiff.imread(original_image_path)
    
    # Perform single FFT analyses
    mask_features, mask_magnitude, mask_processed = fft_analysis(mask, title_prefix="Mask")
    orig_features, orig_magnitude, orig_processed = fft_analysis(image, title_prefix="Original Image")
    
    # Run comprehensive analysis on all images
    print("\n" + "="*80)
    print("RUNNING COMPREHENSIVE COMPARISON ON ALL IMAGES")
    print("="*80)
    
    results = main_comparison_analysis(image_path, mask_path)

# Compare FFt based features with CSV based features

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
from tifffile import imread as tiff_imread
from scipy.signal import find_peaks

def extract_fft_features(mask_path, filename):
    """Extract FFT features from a mask image"""
    # Remove .png extension and add .tif
    base_name = os.path.splitext(filename)[0]  # Remove any extension
    mask_filename = base_name + '.tif'
    mask_image_path = os.path.join(mask_path, mask_filename)
    
    # Check if file exists
    if not os.path.exists(mask_image_path):
        print(f"Mask file not found: {mask_image_path}")
        return None
    
    try:
        mask = tiff_imread(mask_image_path)
        
        # Preprocess mask
        if mask.ndim == 3 and mask.shape[2] == 3:
            mask = mask.mean(axis=2)
        if mask.ndim > 2:
            mask = mask[0]
        mask = mask.astype(np.float32)
        mask = (mask > 0).astype(np.uint8)
        
        # FFT analysis
        f = np.fft.fft2(mask)
        fshift = np.fft.fftshift(f)
        fshift[mask.shape[0]//2, mask.shape[1]//2] = 0
        magnitude = np.abs(fshift)

        h, w = mask.shape
        cy, cx = h//2, w//2

        # Polar coordinates
        Y, X = np.indices((h, w))
        R = np.sqrt((X-cx)**2 + (Y-cy)**2).astype(int)
        theta = np.arctan2(Y-cy, X-cx)
        theta_deg = (np.rad2deg(theta) % 180)

        # Radial profile
        radial_profile = np.bincount(R.ravel(), weights=magnitude.ravel()) / np.bincount(R.ravel())
        r = np.arange(len(radial_profile))

        # Find peaks for density
        peaks, _ = find_peaks(radial_profile, distance=5)
        if len(peaks) > 0:
            main_r = peaks[np.argmax(radial_profile[peaks])]
            density_freq = main_r / w
            spacing_px = 1.0 / density_freq if density_freq > 0 else None
        else:
            main_r = None
            density_freq = None
            spacing_px = None

        # Angular profile
        angular_bins = 180
        hist, edges = np.histogram(theta_deg, bins=angular_bins, weights=magnitude)
        orientation_angle = edges[np.argmax(hist)]
        orientation_angle = (orientation_angle + 90) % 180

        # Alignment and order parameters
        alignment_index = np.max(hist) / np.mean(hist)
        
        if main_r:
            P_peak = radial_profile[main_r]
            mask_bg = (r > main_r + 10) & (r < len(r)//2)
            P_bg = np.mean(radial_profile[mask_bg]) if np.any(mask_bg) else 1
            order_parameter = (P_peak - P_bg) / (P_peak + P_bg)
        else:
            order_parameter = None

        return {
            'fft_density_freq': density_freq,
            'fft_orientation_angle': orientation_angle,
            'fft_alignment_index': alignment_index,
            'fft_order_parameter': order_parameter,
            'fft_spacing_px': spacing_px
        }
    
    except Exception as e:
        print(f"Error processing {mask_filename}: {e}")
        return None

def compute_correlations(csv_path, mask_path):
    """Compute correlations between FFT features and CSV features"""
    
    # Load CSV data
    df = pd.read_csv(csv_path)
    
    # Extract FFT features for each image
    fft_features_list = []
    valid_filenames = []
    
    for filename in df['filename']:
        fft_features = extract_fft_features(mask_path, filename)
        if fft_features is not None:
            fft_features_list.append(fft_features)
            valid_filenames.append(filename)
    
    if len(fft_features_list) == 0:
        print("No valid FFT features extracted. Check file paths and extensions.")
        return None, None
    
    # Create FFT features dataframe
    fft_df = pd.DataFrame(fft_features_list)
    fft_df['filename'] = valid_filenames
    
    # Merge with original data
    merged_df = pd.merge(df, fft_df, on='filename', how='inner')
    
    print(f"Successfully processed {len(merged_df)} images out of {len(df)}")
    
    # Identify CSV feature columns (exclude std columns and filename)
    csv_columns = [col for col in df.columns if col != 'filename']
    # Exclude columns with 'std' in the name
    csv_columns = [col for col in csv_columns if 'std' not in col.lower()]
    
    # FFT feature columns
    fft_columns = [col for col in fft_df.columns if col != 'filename']
    
    # Compute correlations
    results = []
    
    for fft_col in fft_columns:
        for csv_col in csv_columns:
            # Remove rows with NaN values for this pair
            valid_data = merged_df[[fft_col, csv_col]].dropna()
            
            if len(valid_data) > 2:  # Need at least 3 points for correlation
                try:
                    corr, p_value = pearsonr(valid_data[fft_col], valid_data[csv_col])
                    
                    results.append({
                        'fft_feature': fft_col,
                        'csv_feature': csv_col,
                        'correlation': corr,
                        'p_value': p_value,
                        'n_samples': len(valid_data)
                    })
                except Exception as e:
                    print(f"Error computing correlation between {fft_col} and {csv_col}: {e}")
    
    # Create results dataframe
    results_df = pd.DataFrame(results)
    
    return results_df, merged_df

def plot_correlation_heatmap(results_df, figsize=(14, 10)):
    """Plot a heatmap of correlations"""
    if results_df.empty:
        print("No results to plot.")
        return
        
    # Pivot the results
    pivot_table = results_df.pivot(index='csv_feature', 
                                  columns='fft_feature', 
                                  values='correlation')
    
    plt.figure(figsize=figsize)
    sns.heatmap(pivot_table, annot=True, cmap='RdBu_r', center=0,
                fmt='.3f', linewidths=0.5, cbar_kws={'label': 'Pearson Correlation'})
    plt.title('Pearson Correlation Coefficients: FFT vs CSV Features')
    plt.tight_layout()
    plt.show()

def plot_top_correlations(results_df, top_n=10):
    """Plot the top N positive and negative correlations"""
    if results_df.empty:
        print("No results to plot.")
        return
        
    # Separate positive and negative correlations
    pos_corr = results_df[results_df['correlation'] > 0].nlargest(top_n, 'correlation')
    neg_corr = results_df[results_df['correlation'] < 0].nsmallest(top_n, 'correlation')
    
    # Combine and sort
    top_corr = pd.concat([pos_corr, neg_corr]).sort_values('correlation', ascending=False)
    
    # Create plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))
    
    # Positive correlations
    if not pos_corr.empty:
        bars1 = ax1.barh(range(len(pos_corr)), pos_corr['correlation'], color='steelblue')
        ax1.set_yticks(range(len(pos_corr)))
        ax1.set_yticklabels([f"{row['csv_feature']}\nvs {row['fft_feature']}" 
                           for _, row in pos_corr.iterrows()], fontsize=10)
        ax1.set_xlabel('Correlation Coefficient')
        ax1.set_title(f'Top {len(pos_corr)} Positive Correlations')
        ax1.grid(axis='x', alpha=0.3)
        
        # Add correlation values on bars
        for i, (bar, (_, row)) in enumerate(zip(bars1, pos_corr.iterrows())):
            ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                    f'r={row["correlation"]:.3f}\n(p={row["p_value"]:.3f})', 
                    va='center', ha='left', fontsize=9)
    
    # Negative correlations
    if not neg_corr.empty:
        bars2 = ax2.barh(range(len(neg_corr)), neg_corr['correlation'], color='coral')
        ax2.set_yticks(range(len(neg_corr)))
        ax2.set_yticklabels([f"{row['csv_feature']}\nvs {row['fft_feature']}" 
                           for _, row in neg_corr.iterrows()], fontsize=10)
        ax2.set_xlabel('Correlation Coefficient')
        ax2.set_title(f'Top {len(neg_corr)} Negative Correlations')
        ax2.grid(axis='x', alpha=0.3)
        
        # Add correlation values on bars
        for i, (bar, (_, row)) in enumerate(zip(bars2, neg_corr.iterrows())):
            ax2.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2, 
                    f'r={row["correlation"]:.3f}\n(p={row["p_value"]:.3f})', 
                    va='center', ha='right', fontsize=9)
    
    plt.tight_layout()
    plt.show()

def print_detailed_correlations(results_df, feature_groups=None):
    """Print correlations organized by feature groups"""
    if results_df.empty:
        print("No results to display.")
        return
        
    if feature_groups is None:
        feature_groups = {
            'Density Features': ['horizontal_density_cnts_per_um', 'vertical_density_cnts_per_um', 
                               'line_density_horizontal_lines_mean_intersections_per_row',
                               'line_density_vertical_lines_mean_intersections_per_row',
                               'cnt_density_per_um2'],
            'Orientation Features': ['von_mises_mean_orientation_deg', 'nematic_director_angle_deg',
                                   'avg_orientation_angle_deg'],
            'Alignment Features': ['nematic_order_parameter', 'von_mises_concentration_kappa'],
            'Morphology Features': ['avg_length_um', 'avg_width_um', 'avg_area_um2', 
                                  'avg_perimeter_um', 'avg_aspect_ratio', 'num_cnts']
        }
    
    print("DETAILED CORRELATION ANALYSIS")
    print("=" * 80)
    
    for group_name, features in feature_groups.items():
        group_results = results_df[results_df['csv_feature'].isin(features)]
        if not group_results.empty:
            print(f"\n{group_name}:")
            print("-" * 40)
            
            # Display top 3 correlations for each CSV feature in this group
            for csv_feature in features:
                feature_results = group_results[group_results['csv_feature'] == csv_feature]
                if not feature_results.empty:
                    top_3 = feature_results.nlargest(3, 'abs_correlation')
                    print(f"\n  {csv_feature}:")
                    for _, row in top_3.iterrows():
                        sig_star = " ***" if row['p_value'] < 0.001 else " **" if row['p_value'] < 0.01 else " *" if row['p_value'] < 0.05 else ""
                        print(f"    {row['fft_feature']}: r = {row['correlation']:.3f}, p = {row['p_value']:.3f}{sig_star}")

# Main execution
if __name__ == "__main__":
    # Paths - adjust these according to your setup
    csv_path = r"C:\Users\abd93000\Desktop\Projects\CNT\inputdata\new_dataset with test\CNT_analysis_tool\CNT_analysis_tool_test_set\image_level_averages.csv"  # Update this path
    mask_path = "../../data/annotations_uniques/test/masks"  # Your mask path
    
    # Compute correlations
    results_df, merged_df = compute_correlations(csv_path, mask_path)
    
    if results_df is not None and not results_df.empty:
        # Display results
        print("CORRELATION RESULTS SUMMARY")
        print("=" * 80)
        
        # Sort by absolute correlation for easier reading
        results_df['abs_correlation'] = abs(results_df['correlation'])
        sorted_results = results_df.sort_values('abs_correlation', ascending=False)
        
        # Display top correlations
        print(f"\nTop 20 Strongest Correlations (by absolute value):")
        display_cols = ['fft_feature', 'csv_feature', 'correlation', 'p_value', 'n_samples']
        print(sorted_results[display_cols].head(20).to_string(index=False))
        
        # Display statistically significant correlations (p < 0.05)
        significant = results_df[results_df['p_value'] < 0.05].sort_values('abs_correlation', ascending=False)
        print(f"\nStatistically Significant Correlations (p < 0.05): {len(significant)} pairs")
        print(significant[display_cols].head(20).to_string(index=False))
        
        # Print detailed analysis by feature groups
        print_detailed_correlations(results_df)
        
        # Create visualizations
        plot_correlation_heatmap(results_df)
        plot_top_correlations(results_df, top_n=10)
        
        # Save results to CSV
        results_df.to_csv('correlation_results.csv', index=False)
        print(f"\nFull results saved to 'correlation_results.csv'")
        
        # Summary statistics
        print(f"\nSUMMARY STATISTICS:")
        print(f"Total images processed: {len(merged_df)}")
        print(f"Total feature pairs analyzed: {len(results_df)}")
        print(f"Statistically significant pairs (p < 0.05): {len(significant)}")
        print(f"Strong correlations (|r| > 0.7): {len(results_df[results_df['abs_correlation'] > 0.7])}")
        print(f"Moderate correlations (0.5 < |r| <= 0.7): {len(results_df[(results_df['abs_correlation'] > 0.5) & (results_df['abs_correlation'] <= 0.7)])}")
        print(f"Weak correlations (0.3 < |r| <= 0.5): {len(results_df[(results_df['abs_correlation'] > 0.3) & (results_df['abs_correlation'] <= 0.5)])}")
        print(f"Very weak correlations (|r| <= 0.3): {len(results_df[results_df['abs_correlation'] <= 0.3])}")
    
    else:
        print("No correlation results to display. Check your file paths and data.")